In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path



In [5]:
# Define paths
DATA_PATH = Path("../../../../data/raw")

In [6]:
# Load data
df = pd.read_excel(DATA_PATH / "2020_Birth_Final.xlsx")

In [4]:
# ============================================
# INSTALL REQUIRED PACKAGES (if not already installed)
# ============================================
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement
except ImportError:
    print("Installing python-docx...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'python-docx'])
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================
# SET CORRECT COLUMN NAMES
# ============================================

BIRTH_WEIGHT_COL = 'Birth_Weight(grams)'
MOTHER_RACE_COL = 'Race_of_Mother'
FATHER_RACE_COL = 'Race_of_Father'
DISTRICT_COL = 'Registered_District'

print("=" * 80)
print("🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY")
print("=" * 80)

# ============================================
# STEP 1: IDENTIFY MISSING BIRTH WEIGHT COUNTS
# ============================================

print("\n📊 STEP 1: Missing Birth Weight Analysis")
print("-" * 80)

# Check if column exists
if BIRTH_WEIGHT_COL not in df.columns:
    print(f"❌ ERROR: Column '{BIRTH_WEIGHT_COL}' not found!")
    print(f"Available columns: {list(df.columns)}")
else:
    # Create missing birth weight indicator
    df['Missing_Birth_Weight'] = df[BIRTH_WEIGHT_COL].isna()
    
    # Overall missing statistics
    total_records = len(df)
    total_missing = df['Missing_Birth_Weight'].sum()
    total_complete = total_records - total_missing
    
    print(f"\nOverall Statistics:")
    print(f"   • Total records: {total_records:,}")
    print(f"   • Missing birth weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    print(f"   • Complete birth weight: {total_complete:,} ({total_complete/total_records*100:.2f}%)")

# ============================================
# STEP 2: ETHNICITY STANDARDIZATION (IUPAC STANDARDS)
# ============================================

print("\n📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)")
print("-" * 80)

# Define ethnicities with IUPAC codes following international standards
ETHNICITIES = {
    'Sinhalese': {'code': 'SIN', 'full_name': 'Sinhalese', 'iupac_name': 'Sinhala'},
    'Srilankan Tamil': {'code': 'TAM_SL', 'full_name': 'Sri Lankan Tamil', 'iupac_name': 'Tamil (Sri Lanka)'},
    'Indian Tamil': {'code': 'TAM_IN', 'full_name': 'Indian Tamil', 'iupac_name': 'Tamil (India)'},
    'Srilankan Moor': {'code': 'MOOR_SL', 'full_name': 'Sri Lankan Moor', 'iupac_name': 'Moor (Sri Lanka)'},
    'Burgher': {'code': 'BUR', 'full_name': 'Burgher', 'iupac_name': 'Burgher'},
    'Malay': {'code': 'MAL', 'full_name': 'Malay', 'iupac_name': 'Malay'},
    'Srilankan Chetty': {'code': 'CHT_SL', 'full_name': 'Sri Lankan Chetty', 'iupac_name': 'Chetty'},
    'Bharatha': {'code': 'BHA', 'full_name': 'Bharatha', 'iupac_name': 'Bharatha'},
    'Indian Moor': {'code': 'MOOR_IN', 'full_name': 'Indian Moor', 'iupac_name': 'Moor (India)'},
    'Pakistan Moor': {'code': 'MOOR_PK', 'full_name': 'Pakistan Moor', 'iupac_name': 'Moor (Pakistan)'},
    'Other Foreigners': {'code': 'OTH_FGN', 'full_name': 'Other Foreigners', 'iupac_name': 'Other Nationalities'},
    'Other Srilankans': {'code': 'OTH_SL', 'full_name': 'Other Sri Lankans', 'iupac_name': 'Other Ethnic Groups'}
}

# Standardize ethnicity function
def standardize_ethnicity(race):
    """Standardize ethnicity names following IUPAC nomenclature"""
    if pd.isna(race):
        return None
    
    # Convert to string and strip
    race_str = str(race).strip()
    
    # Mapping for numeric codes (1-13) as per Sri Lanka vital statistics
    code_map = {
        '1': 'Sinhalese',
        '2': 'Srilankan Tamil',
        '3': 'Indian Tamil',
        '4': 'Srilankan Moor',
        '5': 'Burgher',
        '6': 'Malay',
        '7': 'Srilankan Chetty',
        '8': 'Bharatha',
        '9': 'Indian Moor',
        '10': 'Pakistan Moor',
        '11': 'Other Foreigners',
        '12': 'Other Srilankans'
    }
    
    # Mapping for text values
    text_map = {
        'Sinhalese': 'Sinhalese',
        'Srilankan Tamil': 'Srilankan Tamil',
        'Sri Lankan Tamil': 'Srilankan Tamil',
        'Indian Tamil': 'Indian Tamil',
        'Srilankan Moor': 'Srilankan Moor',
        'Sri Lankan Moor': 'Srilankan Moor',
        'Moor': 'Srilankan Moor',
        'Burgher': 'Burgher',
        'Malay': 'Malay',
        'Srilankan Chetty': 'Srilankan Chetty',
        'Bharatha': 'Bharatha',
        'Indian Moor': 'Indian Moor',
        'Pakistan Moor': 'Pakistan Moor',
        'Other Foreigners': 'Other Foreigners',
        'Other Srilankans': 'Other Srilankans'
    }
    
    # Check if it's a numeric code
    if race_str in code_map:
        return code_map[race_str]
    
    # Check if it's a text value
    return text_map.get(race_str, None)

# Create standardized ethnicity columns
df['Mother_Ethnicity_Std'] = df[MOTHER_RACE_COL].apply(standardize_ethnicity)
df['Father_Ethnicity_Std'] = df[FATHER_RACE_COL].apply(standardize_ethnicity)

# Show unique ethnicities found
unique_ethnicities = df['Mother_Ethnicity_Std'].dropna().unique()
print(f"\nUnique ethnicities found in dataset: {sorted(unique_ethnicities)}")

# ============================================
# STEP 3: DISTRICT-LEVEL MISSING BIRTH WEIGHT ANALYSIS
# ============================================

print("\n📊 STEP 3: District-Level Missing Birth Weight Analysis")
print("-" * 80)

# Get unique districts
districts = df[DISTRICT_COL].dropna().unique()
district_summary = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    
    total_births = len(district_data)
    missing_bw = district_data['Missing_Birth_Weight'].sum()
    complete_bw = total_births - missing_bw
    
    district_summary.append({
        'District': district,
        'Total_Births': total_births,
        'Missing_BW': missing_bw,
        'Complete_BW': complete_bw,
        'Missing_Rate': (missing_bw / total_births * 100) if total_births > 0 else 0
    })

# Create district summary DataFrame
district_df = pd.DataFrame(district_summary)
district_df = district_df.sort_values('Missing_Rate', ascending=False)

# ============================================
# STEP 4: ETHNICITY DISTRIBUTION BY DISTRICT
# ============================================

print("\n📊 STEP 4: Ethnicity Distribution by District")
print("-" * 80)

# Create detailed ethnicity-district analysis
district_ethnicity_analysis = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    district_total = len(district_data)
    
    if district_total == 0:
        continue
    
    for ethnicity, config in ETHNICITIES.items():
        ethnic_mothers = district_data[district_data['Mother_Ethnicity_Std'] == ethnicity].shape[0]
        ethnic_missing = district_data[
            (district_data['Mother_Ethnicity_Std'] == ethnicity) & 
            (district_data['Missing_Birth_Weight'] == True)
        ].shape[0]
        
        if ethnic_mothers > 0:
            pct_of_district = (ethnic_mothers / district_total * 100)
            pct_of_district_missing = (ethnic_missing / district_total * 100)
            missing_rate_in_ethnic = (ethnic_missing / ethnic_mothers * 100)
            
            district_ethnicity_analysis.append({
                'District': district,
                'Ethnicity': ethnicity,
                'IUPAC_Code': config['code'],
                'IUPAC_Name': config['iupac_name'],
                'Total_Mothers': ethnic_mothers,
                'Pct_of_District_Total': round(pct_of_district, 2),
                'Missing_BW': ethnic_missing,
                'Missing_Rate_in_Ethnic': round(missing_rate_in_ethnic, 2),
                'Pct_of_District_Missing': round(pct_of_district_missing, 2)
            })

# Create DataFrame
if district_ethnicity_analysis:
    ethnicity_district_df = pd.DataFrame(district_ethnicity_analysis)
    
    # ============================================
    # CREATE WORD DOCUMENT WITH IUPAC STANDARDS
    # ============================================
    
    print("\n📄 Creating Word Document with IUPAC Standards...")
    
    # Create new document
    doc = Document()
    
    # Set document margins
    sections = doc.sections
    for section in sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)
    
    # Add title
    title = doc.add_heading('Missing Birth Weight Analysis Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add subtitle with IUPAC standards
    subtitle = doc.add_heading('Sri Lanka Vital Statistics - IUPAC Compliant Nomenclature', 2)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add date
    date_para = doc.add_paragraph(f'Report Generated: {datetime.now().strftime("%B %d, %Y")}')
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 1: EXECUTIVE SUMMARY
    # ============================================
    
    doc.add_heading('1. Executive Summary', level=1)
    
    summary_text = f"""
    This report presents a comprehensive analysis of missing birth weight data in Sri Lanka, 
    stratified by district and ethnicity. The analysis follows IUPAC (International Union of 
    Pure and Applied Chemistry) standards for ethnic nomenclature to ensure international 
    consistency and scientific rigor.
    
    Key Findings:
    • Total Records Analyzed: {total_records:,}
    • Missing Birth Weight Records: {total_missing:,} ({total_missing/total_records*100:.2f}%)
    • Complete Records: {total_complete:,} ({total_complete/total_records*100:.2f}%)
    • Number of Districts Analyzed: {len(districts)}
    • Number of Ethnic Groups Identified: {len(unique_ethnicities)}
    """
    
    doc.add_paragraph(summary_text)
    
    # ============================================
    # SECTION 2: IUPAC ETHNICITY CLASSIFICATION
    # ============================================
    
    doc.add_heading('2. IUPAC Ethnicity Classification System', level=1)
    doc.add_paragraph('The following ethnicity codes follow IUPAC standards for population genetics and vital statistics reporting:')
    
    # Create ethnicity table
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Light Grid Accent 1'
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'IUPAC Standard Name'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
        row_cells[2].text = config['iupac_name']
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 3: DISTRICT-WISE ANALYSIS
    # ============================================
    
    doc.add_heading('3. District-Wise Missing Birth Weight Analysis', level=1)
    
    # Add district summary table
    doc.add_heading('3.1 District Summary Statistics', level=2)
    
    district_table = doc.add_table(rows=1, cols=4)
    district_table.style = 'Light Grid Accent 1'
    hdr_cells = district_table.rows[0].cells
    hdr_cells[0].text = 'District'
    hdr_cells[1].text = 'Total Births'
    hdr_cells[2].text = 'Missing Records'
    hdr_cells[3].text = 'Missing Rate (%)'
    
    for _, row in district_df.iterrows():
        row_cells = district_table.add_row().cells
        row_cells[0].text = row['District']
        row_cells[1].text = f"{row['Total_Births']:,}"
        row_cells[2].text = f"{row['Missing_BW']:,}"
        row_cells[3].text = f"{row['Missing_Rate']:.2f}%"
    
    # Add top districts with highest missing rates
    doc.add_heading('3.2 Districts with Highest Missing Rates', level=2)
    
    top_districts = district_df.head(10)
    top_table = doc.add_table(rows=1, cols=3)
    top_table.style = 'Light Grid Accent 1'
    hdr_cells = top_table.rows[0].cells
    hdr_cells[0].text = 'Rank'
    hdr_cells[1].text = 'District'
    hdr_cells[2].text = 'Missing Rate (%)'
    
    for idx, (_, row) in enumerate(top_districts.iterrows(), 1):
        row_cells = top_table.add_row().cells
        row_cells[0].text = str(idx)
        row_cells[1].text = row['District']
        row_cells[2].text = f"{row['Missing_Rate']:.2f}%"
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 4: DETAILED DISTRICT-ETHNICITY ANALYSIS
    # ============================================
    
    doc.add_heading('4. Detailed District-Ethnicity Analysis', level=1)
    doc.add_paragraph('The following tables show the ethnicity distribution and missing birth weight patterns for each district, following IUPAC nomenclature.')
    
    for district in districts[:15]:  # Show first 15 districts (adjust as needed)
        district_ethnic = ethnicity_district_df[ethnicity_district_df['District'] == district]
        
        if len(district_ethnic) > 0:
            district_total = district_df[district_df['District'] == district]['Total_Births'].values[0]
            district_missing = district_df[district_df['District'] == district]['Missing_BW'].values[0]
            
            # Add district header
            doc.add_heading(f'{district}', level=2)
            doc.add_paragraph(f'Total Births: {district_total:,} | Missing Records: {district_missing:,} ({district_missing/district_total*100:.2f}%)')
            
            # Create ethnicity table for district
            ethnic_table = doc.add_table(rows=1, cols=7)
            ethnic_table.style = 'Light Grid Accent 1'
            hdr_cells = ethnic_table.rows[0].cells
            hdr_cells[0].text = 'IUPAC Code'
            hdr_cells[1].text = 'Ethnicity'
            hdr_cells[2].text = 'Count'
            hdr_cells[3].text = '% of District'
            hdr_cells[4].text = 'Missing'
            hdr_cells[5].text = 'Missing Rate (%)'
            hdr_cells[6].text = '% of District Missing'
            
            district_ethnic_sorted = district_ethnic.sort_values('Pct_of_District_Total', ascending=False)
            
            for _, row in district_ethnic_sorted.iterrows():
                row_cells = ethnic_table.add_row().cells
                row_cells[0].text = row['IUPAC_Code']
                row_cells[1].text = row['Ethnicity']
                row_cells[2].text = f"{row['Total_Mothers']:,}"
                row_cells[3].text = f"{row['Pct_of_District_Total']:.2f}%"
                row_cells[4].text = f"{row['Missing_BW']:,}"
                row_cells[5].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[6].text = f"{row['Pct_of_District_Missing']:.2f}%"
            
            # Add district summary
            doc.add_paragraph()
            most_prevalent = district_ethnic_sorted.iloc[0]
            highest_missing = district_ethnic_sorted.loc[district_ethnic_sorted['Missing_Rate_in_Ethnic'].idxmax()]
            largest_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_District_Missing'].idxmax()]
            
            summary_para = doc.add_paragraph()
            summary_para.add_run('District Summary:').bold = True
            doc.add_paragraph(f'• Most prevalent ethnicity: {most_prevalent["Ethnicity"]} ({most_prevalent["IUPAC_Code"]}) - {most_prevalent["Pct_of_District_Total"]:.1f}% of district')
            doc.add_paragraph(f'• Highest missing rate: {highest_missing["Ethnicity"]} ({highest_missing["IUPAC_Code"]}) - {highest_missing["Missing_Rate_in_Ethnic"]:.1f}% missing')
            doc.add_paragraph(f'• Largest contributor to missing data: {largest_contributor["Ethnicity"]} ({largest_contributor["IUPAC_Code"]}) - {largest_contributor["Pct_of_District_Missing"]:.1f}% of missing records')
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 5: ETHNICITY-SPECIFIC ANALYSIS
    # ============================================
    
    doc.add_heading('5. Ethnicity-Specific Analysis', level=1)
    
    # Calculate overall ethnicity statistics
    ethnicity_overall = ethnicity_district_df.groupby(['Ethnicity', 'IUPAC_Code', 'IUPAC_Name']).agg({
        'Total_Mothers': 'sum',
        'Missing_BW': 'sum'
    }).reset_index()
    ethnicity_overall['Missing_Rate'] = (ethnicity_overall['Missing_BW'] / ethnicity_overall['Total_Mothers'] * 100)
    ethnicity_overall = ethnicity_overall.sort_values('Missing_Rate', ascending=False)
    
    doc.add_heading('5.1 Overall Ethnicity Statistics', level=2)
    
    overall_table = doc.add_table(rows=1, cols=5)
    overall_table.style = 'Light Grid Accent 1'
    hdr_cells = overall_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'Total Births'
    hdr_cells[3].text = 'Missing Records'
    hdr_cells[4].text = 'Missing Rate (%)'
    
    for _, row in ethnicity_overall.iterrows():
        row_cells = overall_table.add_row().cells
        row_cells[0].text = row['IUPAC_Code']
        row_cells[1].text = row['Ethnicity']
        row_cells[2].text = f"{row['Total_Mothers']:,}"
        row_cells[3].text = f"{row['Missing_BW']:,}"
        row_cells[4].text = f"{row['Missing_Rate']:.2f}%"
    
    # Add ethnicity-specific district analysis
    doc.add_heading('5.2 Ethnicity-Specific District Analysis', level=2)
    
    for ethnicity in list(ETHNICITIES.keys())[:6]:  # Show first 6 ethnicities
        ethnic_data = ethnicity_district_df[ethnicity_district_df['Ethnicity'] == ethnicity]
        
        if len(ethnic_data) > 0:
            ethnic_total = ethnic_data['Total_Mothers'].sum()
            ethnic_missing = ethnic_data['Missing_BW'].sum()
            ethnic_rate = (ethnic_missing / ethnic_total * 100) if ethnic_total > 0 else 0
            
            doc.add_heading(f'{ETHNICITIES[ethnicity]["full_name"]} ({ETHNICITIES[ethnicity]["code"]})', level=3)
            doc.add_paragraph(f'Overall Statistics: Total Births: {ethnic_total:,} | Missing: {ethnic_missing:,} ({ethnic_rate:.2f}%)')
            
            # Create table for districts with highest missing rates for this ethnicity
            ethnic_table = doc.add_table(rows=1, cols=4)
            ethnic_table.style = 'Light Grid Accent 1'
            hdr_cells = ethnic_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Records'
            hdr_cells[3].text = 'Missing Rate (%)'
            
            ethnic_sorted = ethnic_data.sort_values('Missing_Rate_in_Ethnic', ascending=False).head(10)
            
            for _, row in ethnic_sorted.iterrows():
                row_cells = ethnic_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_BW']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 6: CONCLUSIONS AND RECOMMENDATIONS
    # ============================================
    
    doc.add_heading('6. Conclusions and Recommendations', level=1)
    
    # Find key insights
    worst_district = district_df.iloc[0]
    worst_ethnicity = ethnicity_overall.iloc[0]
    
    conclusions = f"""
    6.1 Key Findings
    
    • Data Completeness: {total_complete/total_records*100:.2f}% of birth records have complete birth weight data,
      indicating {total_missing/total_records*100:.2f}% of records require data quality improvement.
    
    • Geographic Disparities: {worst_district['District']} shows the highest missing rate at 
      {worst_district['Missing_Rate']:.2f}%, suggesting potential data collection challenges in this region.
    
    • Ethnic Disparities: {worst_ethnicity['Ethnicity']} ({worst_ethnicity['IUPAC_Code']}) has the highest 
      missing rate at {worst_ethnicity['Missing_Rate']:.2f}%, indicating potential systematic bias in data 
      collection across ethnic groups.
    
    6.2 Recommendations
    
    1. Standardize Data Collection Protocols: Implement uniform birth weight recording procedures across all districts,
       particularly in high-missing-rate regions.
    
    2. Ethnicity-Specific Interventions: Develop targeted data quality improvement programs for ethnic groups 
       with high missing rates, respecting cultural and linguistic sensitivities.
    
    3. IUPAC Compliance: Maintain adherence to IUPAC ethnic nomenclature standards to ensure international 
       comparability and scientific rigor.
    
    4. Regular Monitoring: Establish quarterly data quality monitoring systems to track improvements in 
       missing birth weight rates by district and ethnicity.
    
    5. Capacity Building: Provide training for healthcare workers on the importance of complete birth weight 
       documentation, especially in districts with high missing rates.
    """
    
    doc.add_paragraph(conclusions)
    
    # Add methodology section
    doc.add_heading('7. Methodology', level=1)
    
    methodology = f"""
    This analysis was conducted using vital statistics data from Sri Lanka. The methodology follows 
    IUPAC standards for ethnic classification and WHO guidelines for birth weight documentation.
    
    Data Sources:
    • Birth Registration Records: {total_records:,} records
    • Time Period: Full dataset analysis
    • Geographic Coverage: {len(districts)} districts
    
    Ethnic Classification:
    Ethnicities were classified according to IUPAC standards using the following codes:
    """
    
    doc.add_paragraph(methodology)
    
    # Add IUPAC code reference
    ref_table = doc.add_table(rows=1, cols=2)
    ref_table.style = 'Light Grid Accent 1'
    hdr_cells = ref_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnic Group'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = ref_table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
    
    # Save the document
    filename = f'Missing_Birth_Weight_Analysis_IUPAC_{datetime.now().strftime("%Y%m%d_%H%M%S")}.docx'
    doc.save(filename)
    print(f"\n✅ Word document saved as: {filename}")
    
    # ============================================
    # STEP 5: DISPLAY SUMMARY IN CONSOLE
    # ============================================
    
    print("\n" + "=" * 100)
    print("📊 FINAL SUMMARY: Missing Birth Weight Analysis by District and Ethnicity")
    print("=" * 100)
    
    print(f"\n✅ Analysis Complete!")
    print(f"✅ Word Document Generated: {filename}")
    print(f"\nOverall Statistics:")
    print(f"   • Total Districts: {len(districts)}")
    print(f"   • Total Births: {total_records:,}")
    print(f"   • Total Missing Birth Weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    
    print(f"\n🏆 Top 5 Districts with Highest Missing Rate:")
    for _, row in district_df.head(5).iterrows():
        print(f"   • {row['District']}: {row['Missing_Rate']:.2f}% ({row['Missing_BW']:,}/{row['Total_Births']:,})")
    
    print(f"\n🏆 Top 5 Ethnicities with Highest Missing Rate (Overall):")
    for _, row in ethnicity_overall.head(5).iterrows():
        print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Missing_Rate']:.2f}% ({row['Missing_BW']:,}/{row['Total_Mothers']:,})")

else:
    print("\n⚠️ No ethnicity data found. Please check the ethnicity column values.")
    print(f"Sample values from {MOTHER_RACE_COL}:")
    print(df[MOTHER_RACE_COL].value_counts().head(10))

print("\n✅ Analysis Complete!")

🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY

📊 STEP 1: Missing Birth Weight Analysis
--------------------------------------------------------------------------------

Overall Statistics:
   • Total records: 301,712
   • Missing birth weight: 33,542 (11.12%)
   • Complete birth weight: 268,170 (88.88%)

📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)
--------------------------------------------------------------------------------

Unique ethnicities found in dataset: ['Burgher', 'Indian Tamil', 'Malay', 'Sinhalese', 'Srilankan Moor', 'Srilankan Tamil']

📊 STEP 3: District-Level Missing Birth Weight Analysis
--------------------------------------------------------------------------------

📊 STEP 4: Ethnicity Distribution by District
--------------------------------------------------------------------------------

📄 Creating Word Document with IUPAC Standards...

✅ Word document saved as: Missing_Birth_Weight_Analysis_IUPAC_20260329_214519.docx

📊 FINAL SUMMARY: Missin

In [6]:
# ============================================
# INSTALL REQUIRED PACKAGES (if not already installed)
# ============================================
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor, Cm
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement
except ImportError:
    print("Installing python-docx...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'python-docx'])
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor, Cm
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

# Install visualization libraries if needed
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
except ImportError:
    print("Installing visualization libraries...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'matplotlib', 'seaborn'])
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D

import pandas as pd
import numpy as np
from datetime import datetime
import io
import os

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# ============================================
# SET CORRECT COLUMN NAMES
# ============================================

BIRTH_WEIGHT_COL = 'Birth_Weight(grams)'
MOTHER_RACE_COL = 'Race_of_Mother'
FATHER_RACE_COL = 'Race_of_Father'
DISTRICT_COL = 'Registered_District'

print("=" * 80)
print("🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY")
print("=" * 80)

# ============================================
# STEP 1: IDENTIFY MISSING BIRTH WEIGHT COUNTS
# ============================================

print("\n📊 STEP 1: Missing Birth Weight Analysis")
print("-" * 80)

# Check if column exists
if BIRTH_WEIGHT_COL not in df.columns:
    print(f"❌ ERROR: Column '{BIRTH_WEIGHT_COL}' not found!")
    print(f"Available columns: {list(df.columns)}")
else:
    # Create missing birth weight indicator
    df['Missing_Birth_Weight'] = df[BIRTH_WEIGHT_COL].isna()
    
    # Overall missing statistics
    total_records = len(df)
    total_missing = df['Missing_Birth_Weight'].sum()
    total_complete = total_records - total_missing
    
    print(f"\nOverall Statistics:")
    print(f"   • Total records: {total_records:,}")
    print(f"   • Missing birth weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    print(f"   • Complete birth weight: {total_complete:,} ({total_complete/total_records*100:.2f}%)")

# ============================================
# STEP 2: ETHNICITY STANDARDIZATION (IUPAC STANDARDS)
# ============================================

print("\n📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)")
print("-" * 80)

# Define ethnicities with IUPAC codes following international standards
ETHNICITIES = {
    'Sinhalese': {'code': 'SIN', 'full_name': 'Sinhalese', 'iupac_name': 'Sinhala'},
    'Srilankan Tamil': {'code': 'TAM_SL', 'full_name': 'Sri Lankan Tamil', 'iupac_name': 'Tamil (Sri Lanka)'},
    'Indian Tamil': {'code': 'TAM_IN', 'full_name': 'Indian Tamil', 'iupac_name': 'Tamil (India)'},
    'Srilankan Moor': {'code': 'MOOR_SL', 'full_name': 'Sri Lankan Moor', 'iupac_name': 'Moor (Sri Lanka)'},
    'Burgher': {'code': 'BUR', 'full_name': 'Burgher', 'iupac_name': 'Burgher'},
    'Malay': {'code': 'MAL', 'full_name': 'Malay', 'iupac_name': 'Malay'},
    'Srilankan Chetty': {'code': 'CHT_SL', 'full_name': 'Sri Lankan Chetty', 'iupac_name': 'Chetty'},
    'Bharatha': {'code': 'BHA', 'full_name': 'Bharatha', 'iupac_name': 'Bharatha'},
    'Indian Moor': {'code': 'MOOR_IN', 'full_name': 'Indian Moor', 'iupac_name': 'Moor (India)'},
    'Pakistan Moor': {'code': 'MOOR_PK', 'full_name': 'Pakistan Moor', 'iupac_name': 'Moor (Pakistan)'},
    'Other Foreigners': {'code': 'OTH_FGN', 'full_name': 'Other Foreigners', 'iupac_name': 'Other Nationalities'},
    'Other Srilankans': {'code': 'OTH_SL', 'full_name': 'Other Sri Lankans', 'iupac_name': 'Other Ethnic Groups'}
}

# Standardize ethnicity function
def standardize_ethnicity(race):
    """Standardize ethnicity names following IUPAC nomenclature"""
    if pd.isna(race):
        return None
    
    # Convert to string and strip
    race_str = str(race).strip()
    
    # Mapping for numeric codes (1-13) as per Sri Lanka vital statistics
    code_map = {
        '1': 'Sinhalese',
        '2': 'Srilankan Tamil',
        '3': 'Indian Tamil',
        '4': 'Srilankan Moor',
        '5': 'Burgher',
        '6': 'Malay',
        '7': 'Srilankan Chetty',
        '8': 'Bharatha',
        '9': 'Indian Moor',
        '10': 'Pakistan Moor',
        '11': 'Other Foreigners',
        '12': 'Other Srilankans'
    }
    
    # Mapping for text values
    text_map = {
        'Sinhalese': 'Sinhalese',
        'Srilankan Tamil': 'Srilankan Tamil',
        'Sri Lankan Tamil': 'Srilankan Tamil',
        'Indian Tamil': 'Indian Tamil',
        'Srilankan Moor': 'Srilankan Moor',
        'Sri Lankan Moor': 'Srilankan Moor',
        'Moor': 'Srilankan Moor',
        'Burgher': 'Burgher',
        'Malay': 'Malay',
        'Srilankan Chetty': 'Srilankan Chetty',
        'Bharatha': 'Bharatha',
        'Indian Moor': 'Indian Moor',
        'Pakistan Moor': 'Pakistan Moor',
        'Other Foreigners': 'Other Foreigners',
        'Other Srilankans': 'Other Srilankans'
    }
    
    # Check if it's a numeric code
    if race_str in code_map:
        return code_map[race_str]
    
    # Check if it's a text value
    return text_map.get(race_str, None)

# Create standardized ethnicity columns
df['Mother_Ethnicity_Std'] = df[MOTHER_RACE_COL].apply(standardize_ethnicity)
df['Father_Ethnicity_Std'] = df[FATHER_RACE_COL].apply(standardize_ethnicity)

# Show unique ethnicities found
unique_ethnicities = df['Mother_Ethnicity_Std'].dropna().unique()
print(f"\nUnique ethnicities found in dataset: {sorted(unique_ethnicities)}")

# ============================================
# STEP 3: DISTRICT-LEVEL MISSING BIRTH WEIGHT ANALYSIS
# ============================================

print("\n📊 STEP 3: District-Level Missing Birth Weight Analysis")
print("-" * 80)

# Get unique districts
districts = df[DISTRICT_COL].dropna().unique()
district_summary = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    
    total_births = len(district_data)
    missing_bw = district_data['Missing_Birth_Weight'].sum()
    complete_bw = total_births - missing_bw
    
    district_summary.append({
        'District': district,
        'Total_Births': total_births,
        'Missing_BW': missing_bw,
        'Complete_BW': complete_bw,
        'Missing_Rate': (missing_bw / total_births * 100) if total_births > 0 else 0
    })

# Create district summary DataFrame
district_df = pd.DataFrame(district_summary)
district_df = district_df.sort_values('Missing_Rate', ascending=False)

# ============================================
# STEP 4: ETHNICITY DISTRIBUTION BY DISTRICT
# ============================================

print("\n📊 STEP 4: Ethnicity Distribution by District")
print("-" * 80)

# Create detailed ethnicity-district analysis
district_ethnicity_analysis = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    district_total = len(district_data)
    
    if district_total == 0:
        continue
    
    for ethnicity, config in ETHNICITIES.items():
        ethnic_mothers = district_data[district_data['Mother_Ethnicity_Std'] == ethnicity].shape[0]
        ethnic_missing = district_data[
            (district_data['Mother_Ethnicity_Std'] == ethnicity) & 
            (district_data['Missing_Birth_Weight'] == True)
        ].shape[0]
        
        if ethnic_mothers > 0:
            pct_of_district = (ethnic_mothers / district_total * 100)
            pct_of_district_missing = (ethnic_missing / district_total * 100)
            missing_rate_in_ethnic = (ethnic_missing / ethnic_mothers * 100)
            
            district_ethnicity_analysis.append({
                'District': district,
                'Ethnicity': ethnicity,
                'IUPAC_Code': config['code'],
                'IUPAC_Name': config['iupac_name'],
                'Total_Mothers': ethnic_mothers,
                'Pct_of_District_Total': round(pct_of_district, 2),
                'Missing_BW': ethnic_missing,
                'Missing_Rate_in_Ethnic': round(missing_rate_in_ethnic, 2),
                'Pct_of_District_Missing': round(pct_of_district_missing, 2)
            })

# Create DataFrame
if district_ethnicity_analysis:
    ethnicity_district_df = pd.DataFrame(district_ethnicity_analysis)
    
    # Calculate overall ethnicity statistics
    ethnicity_overall = ethnicity_district_df.groupby(['Ethnicity', 'IUPAC_Code', 'IUPAC_Name']).agg({
        'Total_Mothers': 'sum',
        'Missing_BW': 'sum'
    }).reset_index()
    ethnicity_overall['Missing_Rate'] = (ethnicity_overall['Missing_BW'] / ethnicity_overall['Total_Mothers'] * 100)
    ethnicity_overall = ethnicity_overall.sort_values('Missing_Rate', ascending=False)
    
    # ============================================
    # CREATE VISUALIZATIONS
    # ============================================
    
    print("\n📊 Creating visualizations...")
    
    # Create directory for images
    image_dir = "missing_bw_analysis_images"
    os.makedirs(image_dir, exist_ok=True)
    images_paths = []
    
    # FIGURE 1: Overall Missing vs Complete (Pie Chart)
    fig1, ax1 = plt.subplots(figsize=(10, 8))
    sizes = [total_complete, total_missing]
    labels = [f'Complete\n({total_complete:,} births)', f'Missing\n({total_missing:,} births)']
    colors = ['#2ecc71', '#e74c3c']
    explode = (0, 0.05)
    
    wedges, texts, autotexts = ax1.pie(sizes, explode=explode, labels=labels, colors=colors,
                                        autopct='%1.1f%%', startangle=90,
                                        textprops={'fontsize': 12})
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    ax1.set_title('Birth Weight Data Completeness', fontsize=16, fontweight='bold', pad=20)
    
    plt.tight_layout()
    pie_path = os.path.join(image_dir, '01_overall_completeness.png')
    plt.savefig(pie_path, dpi=300, bbox_inches='tight')
    images_paths.append(pie_path)
    plt.close()
    
    # FIGURE 2: Top 15 Districts with Highest Missing Rates (Bar Chart)
    fig2, ax2 = plt.subplots(figsize=(14, 8))
    top15_districts = district_df.head(15).copy()
    
    colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(top15_districts)))
    
    bars = ax2.barh(range(len(top15_districts)), top15_districts['Missing_Rate'].values, color=colors)
    ax2.set_yticks(range(len(top15_districts)))
    ax2.set_yticklabels(top15_districts['District'].values)
    ax2.set_xlabel('Missing Rate (%)', fontsize=12)
    ax2.set_title('Top 15 Districts with Highest Missing Birth Weight Rates', fontsize=16, fontweight='bold')
    
    for i, (bar, rate) in enumerate(zip(bars, top15_districts['Missing_Rate'].values)):
        width = bar.get_width()
        ax2.text(width + 0.5, bar.get_y() + bar.get_height()/2, 
                f'{rate:.1f}%', ha='left', va='center', fontsize=9)
    
    ax2.invert_yaxis()
    ax2.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    
    bar_path = os.path.join(image_dir, '02_top_districts_missing.png')
    plt.savefig(bar_path, dpi=300, bbox_inches='tight')
    images_paths.append(bar_path)
    plt.close()
    
    # FIGURE 3: Ethnicity Missing Rates (Horizontal Bar Chart)
    fig3, ax3 = plt.subplots(figsize=(12, 8))
    
    ethnicity_top = ethnicity_overall.head(12).copy()
    colors_ethnic = plt.cm.RdYlGn_r(np.linspace(0, 1, len(ethnicity_top)))
    
    bars = ax3.barh(range(len(ethnicity_top)), ethnicity_top['Missing_Rate'].values, color=colors_ethnic)
    ax3.set_yticks(range(len(ethnicity_top)))
    ax3.set_yticklabels([f"{row['Ethnicity']} ({row['IUPAC_Code']})" for _, row in ethnicity_top.iterrows()])
    ax3.set_xlabel('Missing Rate (%)', fontsize=12)
    ax3.set_title('Missing Birth Weight Rates by Ethnicity', fontsize=16, fontweight='bold')
    
    for i, (bar, rate) in enumerate(zip(bars, ethnicity_top['Missing_Rate'].values)):
        width = bar.get_width()
        ax3.text(width + 0.5, bar.get_y() + bar.get_height()/2, 
                f'{rate:.1f}%', ha='left', va='center', fontsize=9)
    
    ax3.invert_yaxis()
    ax3.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    
    ethnic_bar_path = os.path.join(image_dir, '03_ethnicity_missing_rates.png')
    plt.savefig(ethnic_bar_path, dpi=300, bbox_inches='tight')
    images_paths.append(ethnic_bar_path)
    plt.close()
    
    # FIGURE 4: District Missing Rate Distribution (Histogram)
    fig4, ax4 = plt.subplots(figsize=(12, 6))
    
    ax4.hist(district_df['Missing_Rate'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    ax4.axvline(district_df['Missing_Rate'].mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {district_df["Missing_Rate"].mean():.1f}%')
    ax4.axvline(district_df['Missing_Rate'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median: {district_df["Missing_Rate"].median():.1f}%')
    
    ax4.set_xlabel('Missing Rate (%)', fontsize=12)
    ax4.set_ylabel('Number of Districts', fontsize=12)
    ax4.set_title('Distribution of Missing Birth Weight Rates Across Districts', fontsize=16, fontweight='bold')
    ax4.legend()
    ax4.grid(alpha=0.3)
    
    plt.tight_layout()
    hist_path = os.path.join(image_dir, '04_district_distribution.png')
    plt.savefig(hist_path, dpi=300, bbox_inches='tight')
    images_paths.append(hist_path)
    plt.close()
    
    # FIGURE 5: Missing Records vs Total Births (Scatter Plot)
    fig5, ax5 = plt.subplots(figsize=(12, 8))
    
    scatter = ax5.scatter(district_df['Total_Births'], district_df['Missing_BW'], 
                         c=district_df['Missing_Rate'], cmap='RdYlGn_r', 
                         s=district_df['Total_Births']/100, alpha=0.6, edgecolors='black', linewidth=0.5)
    
    for _, row in district_df.head(10).iterrows():
        ax5.annotate(row['District'], (row['Total_Births'], row['Missing_BW']),
                    xytext=(5, 5), textcoords='offset points', fontsize=8, alpha=0.7)
    
    ax5.set_xlabel('Total Births', fontsize=12)
    ax5.set_ylabel('Missing Birth Weight Records', fontsize=12)
    ax5.set_title('Missing Records vs Total Births by District\n(Color indicates missing rate)', 
                  fontsize=14, fontweight='bold')
    
    cbar = plt.colorbar(scatter)
    cbar.set_label('Missing Rate (%)', fontsize=10)
    
    ax5.grid(alpha=0.3)
    plt.tight_layout()
    
    scatter_path = os.path.join(image_dir, '05_scatter_missing_vs_total.png')
    plt.savefig(scatter_path, dpi=300, bbox_inches='tight')
    images_paths.append(scatter_path)
    plt.close()
    
    # FIGURE 6: Heatmap of Missing Rates by Ethnicity and District
    fig6, ax6 = plt.subplots(figsize=(14, 10))
    
    top10_districts = district_df.head(10)['District'].values
    top5_ethnicities = ethnicity_overall.head(5)['Ethnicity'].values
    
    heatmap_data = []
    for district in top10_districts:
        row = []
        for ethnicity in top5_ethnicities:
            val = ethnicity_district_df[
                (ethnicity_district_df['District'] == district) & 
                (ethnicity_district_df['Ethnicity'] == ethnicity)
            ]['Missing_Rate_in_Ethnic'].values
            row.append(val[0] if len(val) > 0 else 0)
        heatmap_data.append(row)
    
    heatmap_df = pd.DataFrame(heatmap_data, index=top10_districts, columns=top5_ethnicities)
    
    sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='RdYlGn_r', 
                cbar_kws={'label': 'Missing Rate (%)'}, ax=ax6)
    ax6.set_title('Missing Birth Weight Rates by District and Ethnicity\n(Top 10 Districts with Highest Overall Missing Rates)', 
                  fontsize=14, fontweight='bold')
    ax6.set_xlabel('Ethnicity', fontsize=12)
    ax6.set_ylabel('District', fontsize=12)
    
    plt.tight_layout()
    heatmap_path = os.path.join(image_dir, '06_district_ethnicity_heatmap.png')
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    images_paths.append(heatmap_path)
    plt.close()
    
    # FIGURE 7: Donut Chart - Missing Distribution
    fig7, ax7 = plt.subplots(figsize=(10, 8))
    
    major_ethnicities = ethnicity_overall.head(8)
    other_total = ethnicity_overall.iloc[8:]['Total_Mothers'].sum()
    other_missing = ethnicity_overall.iloc[8:]['Missing_BW'].sum()
    
    if other_total > 0:
        major_ethnicities = pd.concat([
            major_ethnicities,
            pd.DataFrame([{'Ethnicity': 'Others', 'Total_Mothers': other_total, 
                          'Missing_BW': other_missing, 'Missing_Rate': (other_missing/other_total*100)}])
        ], ignore_index=True)
    
    sizes = major_ethnicities['Missing_BW'].values
    labels = [f"{row['Ethnicity']}\n({row['Missing_BW']:,})" for _, row in major_ethnicities.iterrows()]
    colors_donut = plt.cm.Set3(np.linspace(0, 1, len(major_ethnicities)))
    
    wedges, texts, autotexts = ax7.pie(sizes, labels=labels, colors=colors_donut,
                                        autopct='%1.1f%%', startangle=90,
                                        textprops={'fontsize': 10})
    
    centre_circle = plt.Circle((0, 0), 0.70, fc='white')
    fig7.gca().add_artist(centre_circle)
    
    ax7.set_title('Distribution of Missing Birth Weight Records by Ethnicity', 
                  fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    donut_path = os.path.join(image_dir, '07_missing_distribution_donut.png')
    plt.savefig(donut_path, dpi=300, bbox_inches='tight')
    images_paths.append(donut_path)
    plt.close()
    
    # FIGURE 8: Box Plot - Missing Rate Distribution
    fig8, ax8 = plt.subplots(figsize=(12, 6))
    
    boxplot_data = []
    for ethnicity in ethnicity_overall.head(8)['Ethnicity']:
        rates = ethnicity_district_df[ethnicity_district_df['Ethnicity'] == ethnicity]['Missing_Rate_in_Ethnic'].values
        if len(rates) > 0:
            boxplot_data.append(rates)
    
    bp = ax8.boxplot(boxplot_data, labels=ethnicity_overall.head(8)['Ethnicity'].values,
                     patch_artist=True, showmeans=True)
    
    for patch, color in zip(bp['boxes'], plt.cm.Set3(np.linspace(0, 1, len(boxplot_data)))):
        patch.set_facecolor(color)
    
    ax8.set_xlabel('Ethnicity', fontsize=12)
    ax8.set_ylabel('Missing Rate (%)', fontsize=12)
    ax8.set_title('Distribution of Missing Rates Across Districts by Ethnicity', fontsize=14, fontweight='bold')
    ax8.set_xticklabels(ethnicity_overall.head(8)['Ethnicity'].values, rotation=45, ha='right')
    ax8.grid(alpha=0.3)
    
    plt.tight_layout()
    boxplot_path = os.path.join(image_dir, '08_ethnicity_boxplot.png')
    plt.savefig(boxplot_path, dpi=300, bbox_inches='tight')
    images_paths.append(boxplot_path)
    plt.close()
    
    print(f"✅ Created {len(images_paths)} visualizations in '{image_dir}/'")
    
    # ============================================
    # CREATE WORD DOCUMENT WITH IUPAC STANDARDS AND IMAGES
    # ============================================
    
    print("\n📄 Creating Word Document with IUPAC Standards and Visualizations...")
    
    # Create new document
    doc = Document()
    
    # Set document margins
    sections = doc.sections
    for section in sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)
    
    # Add title
    title = doc.add_heading('Missing Birth Weight Analysis Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add subtitle with IUPAC standards
    subtitle = doc.add_heading('Sri Lanka Vital Statistics - IUPAC Compliant Nomenclature', 2)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add date
    date_para = doc.add_paragraph(f'Report Generated: {datetime.now().strftime("%B %d, %Y")}')
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    doc.add_paragraph()
    
    # SECTION 1: EXECUTIVE SUMMARY
    doc.add_heading('1. Executive Summary', level=1)
    
    summary_text = f"""
    This report presents a comprehensive analysis of missing birth weight data in Sri Lanka, 
    stratified by district and ethnicity. The analysis follows IUPAC (International Union of 
    Pure and Applied Chemistry) standards for ethnic nomenclature to ensure international 
    consistency and scientific rigor.
    
    Key Findings:
    • Total Records Analyzed: {total_records:,}
    • Missing Birth Weight Records: {total_missing:,} ({total_missing/total_records*100:.2f}%)
    • Complete Records: {total_complete:,} ({total_complete/total_records*100:.2f}%)
    • Number of Districts Analyzed: {len(districts)}
    • Number of Ethnic Groups Identified: {len(unique_ethnicities)}
    """
    
    doc.add_paragraph(summary_text)
    
    # Add Figure 1: Overall Completeness Pie Chart
    doc.add_heading('Data Completeness Overview', level=2)
    doc.add_picture(pie_path, width=Inches(5))
    doc.add_paragraph('Figure 1: Overall birth weight data completeness.')
    doc.add_paragraph()
    
    # SECTION 2: IUPAC ETHNICITY CLASSIFICATION
    doc.add_heading('2. IUPAC Ethnicity Classification System', level=1)
    doc.add_paragraph('The following ethnicity codes follow IUPAC standards for population genetics and vital statistics reporting:')
    
    # Create ethnicity table
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Light Grid Accent 1'
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'IUPAC Standard Name'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
        row_cells[2].text = config['iupac_name']
    
    doc.add_paragraph()
    
    # SECTION 3: DISTRICT-WISE ANALYSIS
    doc.add_heading('3. District-Wise Missing Birth Weight Analysis', level=1)
    
    # Add Figure 2: Top Districts Bar Chart
    doc.add_heading('Top Districts with Highest Missing Rates', level=2)
    doc.add_picture(bar_path, width=Inches(6))
    doc.add_paragraph('Figure 2: Top 15 districts with highest missing birth weight rates.')
    doc.add_paragraph()
    
    # Add Figure 4: Distribution Histogram
    doc.add_heading('Distribution of Missing Rates', level=2)
    doc.add_picture(hist_path, width=Inches(6))
    doc.add_paragraph('Figure 3: Distribution of missing birth weight rates across all districts.')
    doc.add_paragraph()
    
    # Add district summary table
    doc.add_heading('3.1 District Summary Statistics', level=2)
    
    district_table = doc.add_table(rows=1, cols=4)
    district_table.style = 'Light Grid Accent 1'
    hdr_cells = district_table.rows[0].cells
    hdr_cells[0].text = 'District'
    hdr_cells[1].text = 'Total Births'
    hdr_cells[2].text = 'Missing Records'
    hdr_cells[3].text = 'Missing Rate (%)'
    
    for _, row in district_df.head(20).iterrows():
        row_cells = district_table.add_row().cells
        row_cells[0].text = row['District']
        row_cells[1].text = f"{row['Total_Births']:,}"
        row_cells[2].text = f"{row['Missing_BW']:,}"
        row_cells[3].text = f"{row['Missing_Rate']:.2f}%"
    
    # Add Figure 5: Scatter Plot
    doc.add_heading('Relationship Between Total Births and Missing Records', level=2)
    doc.add_picture(scatter_path, width=Inches(6))
    doc.add_paragraph('Figure 4: Scatter plot showing relationship between total births and missing records.')
    doc.add_page_break()
    
    # SECTION 4: ETHNICITY ANALYSIS
    doc.add_heading('4. Ethnicity-Specific Analysis', level=1)
    
    # Add Figure 3: Ethnicity Bar Chart
    doc.add_heading('Missing Rates by Ethnicity', level=2)
    doc.add_picture(ethnic_bar_path, width=Inches(6))
    doc.add_paragraph('Figure 5: Missing birth weight rates by ethnicity (IUPAC codes shown).')
    doc.add_paragraph()
    
    # Add Figure 7: Donut Chart
    doc.add_picture(donut_path, width=Inches(5))
    doc.add_paragraph('Figure 6: Distribution of missing records across ethnic groups.')
    doc.add_paragraph()
    
    # Add Figure 8: Box Plot
    doc.add_heading('Variability in Missing Rates by Ethnicity', level=2)
    doc.add_picture(boxplot_path, width=Inches(6))
    doc.add_paragraph('Figure 7: Box plot showing distribution of missing rates across districts for each ethnicity.')
    doc.add_paragraph()
    
    # Add overall ethnicity statistics table
    doc.add_heading('4.1 Overall Ethnicity Statistics', level=2)
    
    overall_table = doc.add_table(rows=1, cols=5)
    overall_table.style = 'Light Grid Accent 1'
    hdr_cells = overall_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'Total Births'
    hdr_cells[3].text = 'Missing Records'
    hdr_cells[4].text = 'Missing Rate (%)'
    
    for _, row in ethnicity_overall.iterrows():
        row_cells = overall_table.add_row().cells
        row_cells[0].text = row['IUPAC_Code']
        row_cells[1].text = row['Ethnicity']
        row_cells[2].text = f"{row['Total_Mothers']:,}"
        row_cells[3].text = f"{row['Missing_BW']:,}"
        row_cells[4].text = f"{row['Missing_Rate']:.2f}%"
    
    # Add Figure 6: Heatmap
    doc.add_heading('4.2 District-Ethnicity Interaction Analysis', level=2)
    doc.add_picture(heatmap_path, width=Inches(7))
    doc.add_paragraph('Figure 8: Heatmap showing missing rates by district and ethnicity.')
    doc.add_page_break()
    
    # SECTION 5: CONCLUSIONS AND RECOMMENDATIONS
    doc.add_heading('5. Conclusions and Recommendations', level=1)
    
    # Find key insights
    worst_district = district_df.iloc[0]
    worst_ethnicity = ethnicity_overall.iloc[0]
    
    conclusions = f"""
    5.1 Key Findings
    
    • Data Completeness: {total_complete/total_records*100:.2f}% of birth records have complete birth weight data,
      indicating {total_missing/total_records*100:.2f}% of records require data quality improvement.
    
    • Geographic Disparities: {worst_district['District']} shows the highest missing rate at 
      {worst_district['Missing_Rate']:.2f}%, suggesting potential data collection challenges in this region.
    
    • Ethnic Disparities: {worst_ethnicity['Ethnicity']} ({worst_ethnicity['IUPAC_Code']}) has the highest 
      missing rate at {worst_ethnicity['Missing_Rate']:.2f}%, indicating potential systematic bias in data 
      collection across ethnic groups.
    
    5.2 Recommendations
    
    1. Standardize Data Collection Protocols: Implement uniform birth weight recording procedures across all districts,
       particularly in high-missing-rate regions.
    
    2. Ethnicity-Specific Interventions: Develop targeted data quality improvement programs for ethnic groups 
       with high missing rates, respecting cultural and linguistic sensitivities.
    
    3. IUPAC Compliance: Maintain adherence to IUPAC ethnic nomenclature standards to ensure international 
       comparability and scientific rigor.
    
    4. Regular Monitoring: Establish quarterly data quality monitoring systems to track improvements in 
       missing birth weight rates by district and ethnicity.
    
    5. Capacity Building: Provide training for healthcare workers on the importance of complete birth weight 
       documentation, especially in districts with high missing rates.
    """
    
    doc.add_paragraph(conclusions)
    
    # Save the document
    filename = f'Missing_Birth_Weight_Analysis_IUPAC_{datetime.now().strftime("%Y%m%d_%H%M%S")}.docx'
    doc.save(filename)
    print(f"\n✅ Word document saved as: {filename}")
    
    # ============================================
    # DISPLAY SUMMARY IN CONSOLE
    # ============================================
    
    print("\n" + "=" * 100)
    print("📊 FINAL SUMMARY: Missing Birth Weight Analysis by District and Ethnicity")
    print("=" * 100)
    
    print(f"\n✅ Analysis Complete!")
    print(f"✅ Word Document Generated: {filename}")
    print(f"✅ Visualizations Saved in: {image_dir}/")
    print(f"\nOverall Statistics:")
    print(f"   • Total Districts: {len(districts)}")
    print(f"   • Total Births: {total_records:,}")
    print(f"   • Total Missing Birth Weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    
    print(f"\n🏆 Top 5 Districts with Highest Missing Rate:")
    for _, row in district_df.head(5).iterrows():
        print(f"   • {row['District']}: {row['Missing_Rate']:.2f}% ({row['Missing_BW']:,}/{row['Total_Births']:,})")
    
    print(f"\n🏆 Top 5 Ethnicities with Highest Missing Rate (Overall):")
    for _, row in ethnicity_overall.head(5).iterrows():
        print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Missing_Rate']:.2f}% ({row['Missing_BW']:,}/{row['Total_Mothers']:,})")
    
    print(f"\n📊 Visualizations Created:")
    for img_path in images_paths:
        print(f"   • {os.path.basename(img_path)}")

else:
    print("\n⚠️ No ethnicity data found. Please check the ethnicity column values.")
    print(f"Sample values from {MOTHER_RACE_COL}:")
    print(df[MOTHER_RACE_COL].value_counts().head(10))

print("\n✅ Analysis Complete!")

🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY

📊 STEP 1: Missing Birth Weight Analysis
--------------------------------------------------------------------------------

Overall Statistics:
   • Total records: 301,712
   • Missing birth weight: 33,542 (11.12%)
   • Complete birth weight: 268,170 (88.88%)

📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)
--------------------------------------------------------------------------------

Unique ethnicities found in dataset: ['Burgher', 'Indian Tamil', 'Malay', 'Sinhalese', 'Srilankan Moor', 'Srilankan Tamil']

📊 STEP 3: District-Level Missing Birth Weight Analysis
--------------------------------------------------------------------------------

📊 STEP 4: Ethnicity Distribution by District
--------------------------------------------------------------------------------

📊 Creating visualizations...
✅ Created 8 visualizations in 'missing_bw_analysis_images/'

📄 Creating Word Document with IUPAC Standards and Visualizations...

In [4]:
# ============================================
# INSTALL REQUIRED PACKAGES (if not already installed)
# ============================================
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement
except ImportError:
    print("Installing python-docx...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'python-docx'])
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================
# SET CORRECT COLUMN NAMES
# ============================================

BIRTH_ORDER_COL  = 'Birth_Order'        # Adjust if your column name differs
BIRTH_WEIGHT_COL = 'Birth_Weight(grams)'  # NEW: adjust if your column name differs
MOTHER_RACE_COL  = 'Race_of_Mother'
FATHER_RACE_COL  = 'Race_of_Father'
DISTRICT_COL     = 'Registered_District'

# ============================================
# HELPER: run the identical analysis for any variable
# ============================================

def analyse_missing(df, var_col, var_label, districts, ETHNICITIES, total_records):
    """
    Returns (overall_missing_stats, district_df, ethnicity_district_df)
    for any missingness variable (Birth_Order, Birth_Weight, etc.)
    """
    missing_flag = f'Missing_{var_label}'

    if var_col not in df.columns:
        print(f"  ❌ ERROR: Column '{var_col}' not found!")
        print(f"  Available columns: {list(df.columns)}")
        return None, None, None

    df[missing_flag] = df[var_col].isna()

    total_missing  = int(df[missing_flag].sum())
    total_complete = total_records - total_missing

    print(f"\n  Overall Statistics for {var_label}:")
    print(f"   • Total records       : {total_records:,}")
    print(f"   • Missing {var_label:15s}: {total_missing:,}  ({total_missing/total_records*100:.2f}%)")
    print(f"   • Complete {var_label:13s}: {total_complete:,} ({total_complete/total_records*100:.2f}%)")

    # ---- district summary ----
    district_summary = []
    for district in districts:
        dd = df[df[DISTRICT_COL] == district]
        total_b = len(dd)
        miss    = int(dd[missing_flag].sum())
        district_summary.append({
            'District'           : district,
            'Total_Births'       : total_b,
            f'Missing_{var_label}': miss,
            'Complete'           : total_b - miss,
            'Missing_Rate'       : (miss / total_b * 100) if total_b > 0 else 0,
            'Pct_of_Total_Missing': (miss / total_missing * 100) if total_missing > 0 else 0,
        })

    district_df = pd.DataFrame(district_summary).sort_values('Missing_Rate', ascending=False)

    # ---- district-ethnicity detail ----
    district_ethnicity_analysis = []
    for district in districts:
        dd            = df[df[DISTRICT_COL] == district]
        district_total = len(dd)
        dist_miss_tot  = int(dd[missing_flag].sum())
        if district_total == 0:
            continue

        for ethnicity, config in ETHNICITIES.items():
            eth_mothers = dd[dd['Mother_Ethnicity_Std'] == ethnicity].shape[0]
            eth_missing = dd[
                (dd['Mother_Ethnicity_Std'] == ethnicity) &
                (dd[missing_flag] == True)
            ].shape[0]

            if eth_mothers > 0:
                pct_of_dist        = eth_mothers / district_total * 100
                miss_rate_in_eth   = eth_missing / eth_mothers * 100
                pct_of_dist_miss   = (eth_missing / dist_miss_tot * 100) if dist_miss_tot > 0 else 0
                pct_of_nat_miss    = (eth_missing / total_missing * 100) if total_missing > 0 else 0

                district_ethnicity_analysis.append({
                    'District'                      : district,
                    'Ethnicity'                     : ethnicity,
                    'IUPAC_Code'                    : config['code'],
                    'IUPAC_Name'                    : config['iupac_name'],
                    'Total_Mothers'                 : eth_mothers,
                    'Pct_of_District_Total'         : round(pct_of_dist, 2),
                    f'Missing_{var_label}'          : eth_missing,
                    'Missing_Rate_in_Ethnic'        : round(miss_rate_in_eth, 2),
                    'Pct_of_District_Missing_Records': round(pct_of_dist_miss, 2),
                    'Pct_of_National_Missing'       : round(pct_of_nat_miss, 2),
                })

    ethnicity_district_df = pd.DataFrame(district_ethnicity_analysis) if district_ethnicity_analysis else pd.DataFrame()

    return (
        {'total_missing': total_missing, 'total_complete': total_complete, 'missing_flag': missing_flag},
        district_df,
        ethnicity_district_df,
    )


# ============================================
# HELPER: add one variable's section to the Word doc
# ============================================

def add_variable_section(doc, var_label, stats, district_df, ethnicity_district_df,
                          districts, ETHNICITIES, total_records, section_base_num):
    """
    Writes District, Ethnicity and Conclusions sections for one variable.
    section_base_num is the first heading number to use (e.g. 3 for Birth_Order, 7 for Birth_Weight).
    """
    total_missing = stats['total_missing']
    miss_col      = f'Missing_{var_label}'

    # ---- Section A: District-Wise Analysis ----
    sec_a = section_base_num
    doc.add_heading(f'{sec_a}. District-Wise Missing {var_label} Analysis', level=1)

    doc.add_heading(f'{sec_a}.1 District Summary Statistics', level=2)
    t = doc.add_table(rows=1, cols=6)
    t.style = 'Light Grid Accent 1'
    hdrs = ['District', 'Total Births', f'Missing {var_label}',
            'Missing Rate (%)', '% of Total Missing', 'Rank by Missing Count']
    for i, h in enumerate(hdrs):
        t.rows[0].cells[i].text = h

    by_count = district_df.sort_values(miss_col, ascending=False)
    for idx, (_, row) in enumerate(by_count.iterrows(), 1):
        rc = t.add_row().cells
        rc[0].text = row['District']
        rc[1].text = f"{row['Total_Births']:,}"
        rc[2].text = f"{row[miss_col]:,}"
        rc[3].text = f"{row['Missing_Rate']:.2f}%"
        rc[4].text = f"{row['Pct_of_Total_Missing']:.2f}%"
        rc[5].text = str(idx)

    doc.add_heading(f'{sec_a}.2 Districts with Highest Missing Rates', level=2)
    t2 = doc.add_table(rows=1, cols=4)
    t2.style = 'Light Grid Accent 1'
    for i, h in enumerate(['Rank by Rate', 'District', 'Missing Rate (%)', '% of Total Missing']):
        t2.rows[0].cells[i].text = h
    for idx, (_, row) in enumerate(district_df.head(10).iterrows(), 1):
        rc = t2.add_row().cells
        rc[0].text = str(idx)
        rc[1].text = row['District']
        rc[2].text = f"{row['Missing_Rate']:.2f}%"
        rc[3].text = f"{row['Pct_of_Total_Missing']:.2f}%"

    doc.add_heading(f'{sec_a}.3 Districts with Highest Missing Counts', level=2)
    t3 = doc.add_table(rows=1, cols=4)
    t3.style = 'Light Grid Accent 1'
    for i, h in enumerate(['Rank by Count', 'District', f'Missing {var_label} Count', 'Missing Rate (%)']):
        t3.rows[0].cells[i].text = h
    for idx, (_, row) in enumerate(district_df.nlargest(10, miss_col).iterrows(), 1):
        rc = t3.add_row().cells
        rc[0].text = str(idx)
        rc[1].text = row['District']
        rc[2].text = f"{row[miss_col]:,}"
        rc[3].text = f"{row['Missing_Rate']:.2f}%"

    doc.add_page_break()

    # ---- Section B: Detailed District-Ethnicity ----
    sec_b = section_base_num + 1
    doc.add_heading(f'{sec_b}. Detailed District-Ethnicity Analysis — {var_label}', level=1)
    doc.add_paragraph(
        f'Ethnicity distribution and missing {var_label} patterns for each district, '
        f'following IUPAC nomenclature.'
    )

    if ethnicity_district_df.empty:
        doc.add_paragraph('⚠ No ethnicity data available for this variable.')
    else:
        for district in districts[:15]:
            de = ethnicity_district_df[ethnicity_district_df['District'] == district]
            if len(de) == 0:
                continue

            dist_row        = district_df[district_df['District'] == district].iloc[0]
            district_total  = int(dist_row['Total_Births'])
            district_missing= int(dist_row[miss_col])
            dist_pct        = dist_row['Pct_of_Total_Missing']

            doc.add_heading(district, level=2)
            doc.add_paragraph(
                f'Total Births: {district_total:,} | Missing {var_label}: {district_missing:,} '
                f'({district_missing/district_total*100:.2f}%) | '
                f'{dist_pct:.1f}% of National Missing Records'
            )

            et = doc.add_table(rows=1, cols=9)
            et.style = 'Light Grid Accent 1'
            eh = ['IUPAC Code', 'Ethnicity', 'Count', '% of District',
                  f'Missing {var_label}', 'Missing Rate (%)',
                  '% of District Missing', '% of National Missing', 'Rank in District']
            for i, h in enumerate(eh):
                et.rows[0].cells[i].text = h

            de_sorted = de.sort_values('Pct_of_District_Total', ascending=False).copy()
            de_sorted['Rank_In_District'] = (
                de_sorted[miss_col].rank(ascending=False, method='dense').astype(int)
            )

            for _, row in de_sorted.iterrows():
                rc = et.add_row().cells
                rc[0].text = row['IUPAC_Code']
                rc[1].text = row['Ethnicity']
                rc[2].text = f"{row['Total_Mothers']:,}"
                rc[3].text = f"{row['Pct_of_District_Total']:.2f}%"
                rc[4].text = f"{row[miss_col]:,}"
                rc[5].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                rc[6].text = f"{row['Pct_of_District_Missing_Records']:.2f}%"
                rc[7].text = f"{row['Pct_of_National_Missing']:.2f}%"
                rc[8].text = str(row['Rank_In_District'])

            doc.add_paragraph()
            most_prev   = de_sorted.iloc[0]
            high_miss   = de_sorted.loc[de_sorted['Missing_Rate_in_Ethnic'].idxmax()]
            lrg_dist    = de_sorted.loc[de_sorted['Pct_of_District_Missing_Records'].idxmax()]
            lrg_nat     = de_sorted.loc[de_sorted['Pct_of_National_Missing'].idxmax()]

            doc.add_paragraph().add_run('District Summary:').bold = True
            for bullet in [
                f"Most prevalent ethnicity: {most_prev['Ethnicity']} ({most_prev['IUPAC_Code']}) — {most_prev['Pct_of_District_Total']:.1f}% of district",
                f"Highest missing rate: {high_miss['Ethnicity']} ({high_miss['IUPAC_Code']}) — {high_miss['Missing_Rate_in_Ethnic']:.1f}% missing",
                f"Largest contributor to district missing data: {lrg_dist['Ethnicity']} ({lrg_dist['IUPAC_Code']}) — {lrg_dist['Pct_of_District_Missing_Records']:.1f}% of district missing records",
                f"Largest contributor to national missing data: {lrg_nat['Ethnicity']} ({lrg_nat['IUPAC_Code']}) — {lrg_nat['Pct_of_National_Missing']:.1f}% of national missing records",
            ]:
                doc.add_paragraph(f'• {bullet}')
            doc.add_paragraph()

    doc.add_page_break()

    # ---- Section C: Ethnicity-Specific ----
    sec_c = section_base_num + 2
    doc.add_heading(f'{sec_c}. Ethnicity-Specific Analysis — {var_label}', level=1)

    if not ethnicity_district_df.empty:
        eth_overall = ethnicity_district_df.groupby(
            ['Ethnicity', 'IUPAC_Code', 'IUPAC_Name']
        ).agg({
            'Total_Mothers'        : 'sum',
            miss_col               : 'sum',
            'Pct_of_National_Missing': 'sum',
        }).reset_index()
        eth_overall['Missing_Rate'] = eth_overall[miss_col] / eth_overall['Total_Mothers'] * 100
        eth_overall = eth_overall.sort_values('Missing_Rate', ascending=False)

        doc.add_heading(f'{sec_c}.1 Overall Ethnicity Statistics', level=2)
        ot = doc.add_table(rows=1, cols=6)
        ot.style = 'Light Grid Accent 1'
        for i, h in enumerate(['IUPAC Code', 'Ethnicity', 'Total Births',
                                f'Missing {var_label}', 'Missing Rate (%)', '% of National Missing']):
            ot.rows[0].cells[i].text = h
        for _, row in eth_overall.iterrows():
            rc = ot.add_row().cells
            rc[0].text = row['IUPAC_Code']
            rc[1].text = row['Ethnicity']
            rc[2].text = f"{row['Total_Mothers']:,}"
            rc[3].text = f"{row[miss_col]:,}"
            rc[4].text = f"{row['Missing_Rate']:.2f}%"
            rc[5].text = f"{row['Pct_of_National_Missing']:.2f}%"

        doc.add_heading(f'{sec_c}.2 Ethnicity-Specific District Analysis', level=2)

        for ethnicity in list(ETHNICITIES.keys())[:6]:
            ed = ethnicity_district_df[ethnicity_district_df['Ethnicity'] == ethnicity]
            if len(ed) == 0:
                continue

            eth_total   = ed['Total_Mothers'].sum()
            eth_missing = ed[miss_col].sum()
            eth_rate    = (eth_missing / eth_total * 100) if eth_total > 0 else 0
            eth_nat_pct = ed['Pct_of_National_Missing'].sum()

            doc.add_heading(
                f"{ETHNICITIES[ethnicity]['full_name']} ({ETHNICITIES[ethnicity]['code']})", level=3
            )
            doc.add_paragraph(
                f'Overall Statistics: Total Births: {eth_total:,} | '
                f'Missing {var_label}: {eth_missing:,} ({eth_rate:.2f}%) | '
                f'{eth_nat_pct:.1f}% of National Missing Records'
            )

            for sort_col, label in [
                ('Missing_Rate_in_Ethnic', 'Highest Missing Rates'),
                (miss_col,                 'Highest Missing Counts'),
            ]:
                doc.add_paragraph(
                    f'Districts with {label} for this Ethnicity:', style='List Bullet'
                )
                rt = doc.add_table(rows=1, cols=5)
                rt.style = 'Light Grid Accent 1'
                for i, h in enumerate(['District', 'Total Births', f'Missing {var_label}',
                                        'Missing Rate (%)', '% of Ethnic Missing']):
                    rt.rows[0].cells[i].text = h
                for _, row in ed.sort_values(sort_col, ascending=False).head(10).iterrows():
                    rc = rt.add_row().cells
                    rc[0].text = row['District']
                    rc[1].text = f"{row['Total_Mothers']:,}"
                    rc[2].text = f"{row[miss_col]:,}"
                    rc[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                    rc[4].text = f"{(row[miss_col]/eth_missing*100):.2f}%" if eth_missing > 0 else '0.00%'
                doc.add_paragraph()

    doc.add_page_break()

    # ---- Section D: Conclusions ----
    sec_d = section_base_num + 3
    doc.add_heading(f'{sec_d}. Conclusions and Recommendations — {var_label}', level=1)

    worst_by_rate  = district_df.iloc[0]
    worst_by_count = district_df.nlargest(1, miss_col).iloc[0]

    if not ethnicity_district_df.empty:
        eth_overall_tmp = ethnicity_district_df.groupby(['Ethnicity','IUPAC_Code']).agg(
            {miss_col: 'sum', 'Total_Mothers': 'sum', 'Pct_of_National_Missing': 'sum'}
        ).reset_index()
        eth_overall_tmp['Missing_Rate'] = eth_overall_tmp[miss_col] / eth_overall_tmp['Total_Mothers'] * 100
        worst_eth  = eth_overall_tmp.sort_values('Missing_Rate', ascending=False).iloc[0]
        top_nat    = eth_overall_tmp.nlargest(1, 'Pct_of_National_Missing').iloc[0]
        eth_bullets = (
            f"• Ethnic Disparities by Rate: {worst_eth['Ethnicity']} ({worst_eth['IUPAC_Code']}) "
            f"has the highest missing rate for {var_label} at {worst_eth['Missing_Rate']:.2f}%.\n\n"
            f"• Ethnic Disparities by Contribution: {top_nat['Ethnicity']} ({top_nat['IUPAC_Code']}) "
            f"contributes the largest share ({top_nat['Pct_of_National_Missing']:.1f}%) of all "
            f"missing {var_label} records nationally."
        )
        rec_eth = (
            f"3. Ethnicity-Specific Interventions: Develop targeted data quality improvement programs "
            f"for ethnic groups with both high missing rates ({worst_eth['Ethnicity']}: "
            f"{worst_eth['Missing_Rate']:.1f}%) and high contribution to national missing "
            f"({top_nat['Ethnicity']}: {top_nat['Pct_of_National_Missing']:.1f}%)."
        )
    else:
        eth_bullets = ''
        rec_eth     = '3. Ethnicity-specific analysis unavailable — check ethnicity column values.'

    total_complete = total_records - total_missing
    conclusions = f"""
{sec_d}.1 Key Findings

• Data Completeness: {total_complete/total_records*100:.2f}% of birth records have complete {var_label} data,
  indicating {total_missing/total_records*100:.2f}% of records require data quality improvement.

• Geographic Disparities by Rate: {worst_by_rate['District']} shows the highest missing rate for
  {var_label} at {worst_by_rate['Missing_Rate']:.2f}%.

• Geographic Disparities by Count: {worst_by_count['District']} has the highest absolute number of
  missing records ({worst_by_count[miss_col]:,} records, representing
  {worst_by_count['Pct_of_Total_Missing']:.1f}% of all missing data).

{eth_bullets}

{sec_d}.2 Recommendations

1. Prioritise High-Volume Districts: Focus data quality improvement on districts with the highest
   absolute missing counts ({worst_by_count['District']}, {worst_by_count[miss_col]:,} missing records).

2. Target High-Rate Districts: Implement specialised interventions in districts with the highest
   missing rates ({worst_by_rate['District']}: {worst_by_rate['Missing_Rate']:.1f}% missing).

{rec_eth}

4. IUPAC Compliance: Maintain adherence to IUPAC ethnic nomenclature standards.

5. Regular Monitoring: Establish quarterly data quality monitoring to track improvements in
   missing {var_label} rates by district and ethnicity.

6. Capacity Building: Provide training for healthcare workers on complete {var_label} documentation,
   especially in high-volume and high-rate districts.

7. Electronic Health Records: Implement or strengthen EHR systems with mandatory {var_label}
   fields to reduce missing data.
"""
    doc.add_paragraph(conclusions)
    doc.add_page_break()


# ============================================================
# MAIN ANALYSIS
# ============================================================

print("=" * 80)
print("🏥 MISSING DATA ANALYSIS: BIRTH ORDER & BIRTH WEIGHT BY DISTRICT AND ETHNICITY")
print("=" * 80)

# ============================================
# STEP 1: ETHNICITY STANDARDIZATION
# ============================================

print("\n📊 STEP 1: Standardising Ethnicities (IUPAC Standards)")
print("-" * 80)

ETHNICITIES = {
    'Sinhalese'       : {'code': 'SIN',      'full_name': 'Sinhalese',          'iupac_name': 'Sinhala'},
    'Srilankan Tamil' : {'code': 'TAM_SL',   'full_name': 'Sri Lankan Tamil',   'iupac_name': 'Tamil (Sri Lanka)'},
    'Indian Tamil'    : {'code': 'TAM_IN',   'full_name': 'Indian Tamil',        'iupac_name': 'Tamil (India)'},
    'Srilankan Moor'  : {'code': 'MOOR_SL',  'full_name': 'Sri Lankan Moor',    'iupac_name': 'Moor (Sri Lanka)'},
    'Burgher'         : {'code': 'BUR',      'full_name': 'Burgher',             'iupac_name': 'Burgher'},
    'Malay'           : {'code': 'MAL',      'full_name': 'Malay',               'iupac_name': 'Malay'},
    'Srilankan Chetty': {'code': 'CHT_SL',   'full_name': 'Sri Lankan Chetty',  'iupac_name': 'Chetty'},
    'Bharatha'        : {'code': 'BHA',      'full_name': 'Bharatha',            'iupac_name': 'Bharatha'},
    'Indian Moor'     : {'code': 'MOOR_IN',  'full_name': 'Indian Moor',        'iupac_name': 'Moor (India)'},
    'Pakistan Moor'   : {'code': 'MOOR_PK',  'full_name': 'Pakistan Moor',      'iupac_name': 'Moor (Pakistan)'},
    'Other Foreigners': {'code': 'OTH_FGN',  'full_name': 'Other Foreigners',   'iupac_name': 'Other Nationalities'},
    'Other Srilankans': {'code': 'OTH_SL',   'full_name': 'Other Sri Lankans',  'iupac_name': 'Other Ethnic Groups'},
}

def standardize_ethnicity(race):
    if pd.isna(race):
        return None
    race_str = str(race).strip()
    code_map = {
        '1': 'Sinhalese', '2': 'Srilankan Tamil', '3': 'Indian Tamil',
        '4': 'Srilankan Moor', '5': 'Burgher', '6': 'Malay',
        '7': 'Srilankan Chetty', '8': 'Bharatha', '9': 'Indian Moor',
        '10': 'Pakistan Moor', '11': 'Other Foreigners', '12': 'Other Srilankans',
    }
    text_map = {
        'Sinhalese': 'Sinhalese', 'Srilankan Tamil': 'Srilankan Tamil',
        'Sri Lankan Tamil': 'Srilankan Tamil', 'Indian Tamil': 'Indian Tamil',
        'Srilankan Moor': 'Srilankan Moor', 'Sri Lankan Moor': 'Srilankan Moor',
        'Moor': 'Srilankan Moor', 'Burgher': 'Burgher', 'Malay': 'Malay',
        'Srilankan Chetty': 'Srilankan Chetty', 'Bharatha': 'Bharatha',
        'Indian Moor': 'Indian Moor', 'Pakistan Moor': 'Pakistan Moor',
        'Other Foreigners': 'Other Foreigners', 'Other Srilankans': 'Other Srilankans',
    }
    if race_str in code_map:
        return code_map[race_str]
    return text_map.get(race_str, None)

df['Mother_Ethnicity_Std'] = df[MOTHER_RACE_COL].apply(standardize_ethnicity)
df['Father_Ethnicity_Std'] = df[FATHER_RACE_COL].apply(standardize_ethnicity)

unique_ethnicities = df['Mother_Ethnicity_Std'].dropna().unique()
print(f"\nUnique ethnicities found: {sorted(unique_ethnicities)}")

total_records = len(df)
districts     = df[DISTRICT_COL].dropna().unique()

# ============================================
# STEP 2: ANALYSE BOTH VARIABLES
# ============================================

VARIABLES = [
    (BIRTH_ORDER_COL,  'Birth_Order'),
    (BIRTH_WEIGHT_COL, 'Birth_Weight'),  # NEW
]

results = {}
for col, label in VARIABLES:
    print(f"\n{'='*80}")
    print(f"📊 Analysing Missing: {label}")
    print(f"{'='*80}")
    stats, dist_df, eth_dist_df = analyse_missing(df, col, label, districts, ETHNICITIES, total_records)
    if stats is not None:
        results[label] = {
            'stats'             : stats,
            'district_df'       : dist_df,
            'ethnicity_dist_df' : eth_dist_df,
        }

# ============================================
# STEP 3: CREATE WORD DOCUMENT
# ============================================

if results:
    print("\n📄 Creating Word Document...")

    doc = Document()

    # Margins
    for section in doc.sections:
        section.top_margin    = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin   = Inches(1)
        section.right_margin  = Inches(1)

    # Title
    t = doc.add_heading('Missing Data Analysis Report: Birth Order & Birth Weight', 0)
    t.alignment = WD_ALIGN_PARAGRAPH.CENTER

    s = doc.add_heading('Sri Lanka Vital Statistics — IUPAC Compliant Nomenclature', 2)
    s.alignment = WD_ALIGN_PARAGRAPH.CENTER

    dp = doc.add_paragraph(f'Report Generated: {datetime.now().strftime("%B %d, %Y")}')
    dp.alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph()

    # Section 1: Executive Summary
    doc.add_heading('1. Executive Summary', level=1)

    bo_stats = results.get('Birth_Order', {}).get('stats', {})
    bw_stats = results.get('Birth_Weight', {}).get('stats', {})

    summary = f"""
    This report presents a comprehensive analysis of two key missing data variables in Sri Lanka
    vital birth statistics: Birth Order and Birth Weight (grams). The analysis is stratified by
    district and ethnicity and follows IUPAC standards for ethnic nomenclature.

    Records Analysed: {total_records:,}
    Districts Covered: {len(districts)}
    Ethnic Groups Identified: {len(unique_ethnicities)}

    Birth Order
    • Missing: {bo_stats.get('total_missing', 'N/A'):,} ({bo_stats.get('total_missing', 0)/total_records*100:.2f}%)
    • Complete: {bo_stats.get('total_complete', 'N/A'):,} ({bo_stats.get('total_complete', 0)/total_records*100:.2f}%)

    Birth Weight (grams)
    • Missing: {bw_stats.get('total_missing', 'N/A'):,} ({bw_stats.get('total_missing', 0)/total_records*100:.2f}%)
    • Complete: {bw_stats.get('total_complete', 'N/A'):,} ({bw_stats.get('total_complete', 0)/total_records*100:.2f}%)
    """
    doc.add_paragraph(summary)

    # Section 2: IUPAC Classification
    doc.add_heading('2. IUPAC Ethnicity Classification System', level=1)
    doc.add_paragraph('The following codes follow IUPAC standards for population genetics and vital statistics reporting:')
    et = doc.add_table(rows=1, cols=3)
    et.style = 'Light Grid Accent 1'
    for i, h in enumerate(['IUPAC Code', 'Ethnicity', 'IUPAC Standard Name']):
        et.rows[0].cells[i].text = h
    for eth, cfg in ETHNICITIES.items():
        rc = et.add_row().cells
        rc[0].text = cfg['code']
        rc[1].text = cfg['full_name']
        rc[2].text = cfg['iupac_name']
    doc.add_paragraph()

    # Sections 3-6: Birth Order  (section numbers 3, 4, 5, 6)
    if 'Birth_Order' in results:
        add_variable_section(
            doc, 'Birth_Order',
            results['Birth_Order']['stats'],
            results['Birth_Order']['district_df'],
            results['Birth_Order']['ethnicity_dist_df'],
            districts, ETHNICITIES, total_records,
            section_base_num=3,
        )

    # Sections 7-10: Birth Weight  (section numbers 7, 8, 9, 10)
    if 'Birth_Weight' in results:
        add_variable_section(
            doc, 'Birth_Weight',
            results['Birth_Weight']['stats'],
            results['Birth_Weight']['district_df'],
            results['Birth_Weight']['ethnicity_dist_df'],
            districts, ETHNICITIES, total_records,
            section_base_num=7,
        )

    # Section 11: Methodology
    doc.add_heading('11. Methodology', level=1)
    methodology = f"""
    Data Sources:
    • Birth Registration Records: {total_records:,} records
    • Geographic Coverage: {len(districts)} districts

    Variables Analysed:
    • Birth Order — numerical order of a child's birth among all live births to the same mother.
    • Birth Weight (grams) — recorded weight of the newborn in grams at time of birth.

    Key Metrics Defined:

    1. Missing Rate: (Missing Records / Total Births) × 100

    2. Percentage of Total Missing Records:
       (Missing in Group / Total National Missing) × 100

    3. Percentage of District Missing Records:
       (Missing for Ethnicity in District / Total District Missing) × 100

    4. Percentage of National Missing:
       (Missing for Ethnicity-District / Total National Missing) × 100

    Ethnic Classification follows IUPAC standards (codes listed in Section 2).
    """
    doc.add_paragraph(methodology)

    rt = doc.add_table(rows=1, cols=2)
    rt.style = 'Light Grid Accent 1'
    rt.rows[0].cells[0].text = 'IUPAC Code'
    rt.rows[0].cells[1].text = 'Ethnic Group'
    for eth, cfg in ETHNICITIES.items():
        rc = rt.add_row().cells
        rc[0].text = cfg['code']
        rc[1].text = cfg['full_name']

    # Save
    filename = f'Missing_BirthOrder_BirthWeight_Analysis_IUPAC_{datetime.now().strftime("%Y%m%d_%H%M%S")}.docx'
    doc.save(filename)
    print(f"\n✅ Word document saved as: {filename}")

    # ============================================
    # STEP 4: CONSOLE SUMMARY
    # ============================================

    for label, res in results.items():
        d_df    = res['district_df']
        miss_col = f'Missing_{label}'

        print(f"\n{'='*100}")
        print(f"📊 SUMMARY: Missing {label}")
        print(f"{'='*100}")
        total_miss = res['stats']['total_missing']
        print(f"   Total Missing: {total_miss:,} ({total_miss/total_records*100:.2f}%)")

        print(f"\n🏆 Top 5 Districts — Highest Missing RATE:")
        for _, row in d_df.head(5).iterrows():
            print(f"   • {row['District']}: {row['Missing_Rate']:.2f}% ({row[miss_col]:,}/{row['Total_Births']:,})")

        print(f"\n🏆 Top 5 Districts — Highest Missing COUNT:")
        for _, row in d_df.nlargest(5, miss_col).iterrows():
            print(f"   • {row['District']}: {row[miss_col]:,} records ({row['Missing_Rate']:.2f}%)")

        print(f"\n🏆 Top 5 Districts — % of National Missing:")
        for _, row in d_df.nlargest(5, 'Pct_of_Total_Missing').iterrows():
            print(f"   • {row['District']}: {row['Pct_of_Total_Missing']:.2f}% ({row[miss_col]:,} records)")

        eth_dist_df = res['ethnicity_dist_df']
        if not eth_dist_df.empty:
            eth_sum = eth_dist_df.groupby(['Ethnicity', 'IUPAC_Code']).agg(
                {miss_col: 'sum', 'Total_Mothers': 'sum', 'Pct_of_National_Missing': 'sum'}
            ).reset_index()
            eth_sum['Missing_Rate'] = eth_sum[miss_col] / eth_sum['Total_Mothers'] * 100

            print(f"\n🏆 Top 5 Ethnicities — Highest Missing RATE:")
            for _, row in eth_sum.sort_values('Missing_Rate', ascending=False).head(5).iterrows():
                print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Missing_Rate']:.2f}%")

            print(f"\n🏆 Top 5 Ethnicities — % of National Missing:")
            for _, row in eth_sum.nlargest(5, 'Pct_of_National_Missing').iterrows():
                print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Pct_of_National_Missing']:.2f}%")

else:
    print("\n⚠️ No valid variable columns found. Please check column names at the top of the script.")

print("\n✅ Analysis Complete!")

🏥 MISSING DATA ANALYSIS: BIRTH ORDER & BIRTH WEIGHT BY DISTRICT AND ETHNICITY

📊 STEP 1: Standardising Ethnicities (IUPAC Standards)
--------------------------------------------------------------------------------

Unique ethnicities found: ['Burgher', 'Indian Tamil', 'Malay', 'Sinhalese', 'Srilankan Moor', 'Srilankan Tamil']

📊 Analysing Missing: Birth_Order

  Overall Statistics for Birth_Order:
   • Total records       : 301,712
   • Missing Birth_Order    : 27,218  (9.02%)
   • Complete Birth_Order  : 274,494 (90.98%)

📊 Analysing Missing: Birth_Weight

  Overall Statistics for Birth_Weight:
   • Total records       : 301,712
   • Missing Birth_Weight   : 33,542  (11.12%)
   • Complete Birth_Weight : 268,170 (88.88%)

📄 Creating Word Document...

✅ Word document saved as: Missing_BirthOrder_BirthWeight_Analysis_IUPAC_20260330_060302.docx

📊 SUMMARY: Missing Birth_Order
   Total Missing: 27,218 (9.02%)

🏆 Top 5 Districts — Highest Missing RATE:
   • Batticaloa: 79.97% (7,650/9,566)


In [7]:
# ============================================
# INSTALL REQUIRED PACKAGES (if not already installed)
# ============================================
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement
except ImportError:
    print("Installing python-docx...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'python-docx'])
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================
# SET CORRECT COLUMN NAMES
# ============================================

# CHANGED: Birth Weight column name (adjust based on your actual column name)
BIRTH_WEIGHT_COL = 'Birth_Weight(grams)'  # Change this to your actual column name
MOTHER_RACE_COL = 'Race_of_Mother'
FATHER_RACE_COL = 'Race_of_Father'
DISTRICT_COL = 'Registered_District'

print("=" * 80)
print("🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY")
print("=" * 80)

# ============================================
# STEP 1: IDENTIFY MISSING BIRTH WEIGHT COUNTS
# ============================================

print("\n📊 STEP 1: Missing Birth Weight Analysis")
print("-" * 80)

# Check if column exists
if BIRTH_WEIGHT_COL not in df.columns:
    print(f"❌ ERROR: Column '{BIRTH_WEIGHT_COL}' not found!")
    print(f"Available columns: {list(df.columns)}")
    print(f"\n💡 TIP: Please update BIRTH_WEIGHT_COL with the correct column name")
else:
    # Create missing birth weight indicator
    df['Missing_Birth_Weight'] = df[BIRTH_WEIGHT_COL].isna()
    
    # Overall missing statistics
    total_records = len(df)
    total_missing = df['Missing_Birth_Weight'].sum()
    total_complete = total_records - total_missing
    
    print(f"\nOverall Statistics:")
    print(f"   • Total records: {total_records:,}")
    print(f"   • Missing birth weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    print(f"   • Complete birth weight: {total_complete:,} ({total_complete/total_records*100:.2f}%)")

# ============================================
# STEP 2: ETHNICITY STANDARDIZATION (IUPAC STANDARDS)
# ============================================

print("\n📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)")
print("-" * 80)

# Define ethnicities with IUPAC codes following international standards
ETHNICITIES = {
    'Sinhalese': {'code': 'SIN', 'full_name': 'Sinhalese', 'iupac_name': 'Sinhala'},
    'Srilankan Tamil': {'code': 'TAM_SL', 'full_name': 'Sri Lankan Tamil', 'iupac_name': 'Tamil (Sri Lanka)'},
    'Indian Tamil': {'code': 'TAM_IN', 'full_name': 'Indian Tamil', 'iupac_name': 'Tamil (India)'},
    'Srilankan Moor': {'code': 'MOOR_SL', 'full_name': 'Sri Lankan Moor', 'iupac_name': 'Moor (Sri Lanka)'},
    'Burgher': {'code': 'BUR', 'full_name': 'Burgher', 'iupac_name': 'Burgher'},
    'Malay': {'code': 'MAL', 'full_name': 'Malay', 'iupac_name': 'Malay'},
    'Srilankan Chetty': {'code': 'CHT_SL', 'full_name': 'Sri Lankan Chetty', 'iupac_name': 'Chetty'},
    'Bharatha': {'code': 'BHA', 'full_name': 'Bharatha', 'iupac_name': 'Bharatha'},
    'Indian Moor': {'code': 'MOOR_IN', 'full_name': 'Indian Moor', 'iupac_name': 'Moor (India)'},
    'Pakistan Moor': {'code': 'MOOR_PK', 'full_name': 'Pakistan Moor', 'iupac_name': 'Moor (Pakistan)'},
    'Other Foreigners': {'code': 'OTH_FGN', 'full_name': 'Other Foreigners', 'iupac_name': 'Other Nationalities'},
    'Other Srilankans': {'code': 'OTH_SL', 'full_name': 'Other Sri Lankans', 'iupac_name': 'Other Ethnic Groups'}
}

# Standardize ethnicity function
def standardize_ethnicity(race):
    """Standardize ethnicity names following IUPAC nomenclature"""
    if pd.isna(race):
        return None
    
    # Convert to string and strip
    race_str = str(race).strip()
    
    # Mapping for numeric codes (1-13) as per Sri Lanka vital statistics
    code_map = {
        '1': 'Sinhalese',
        '2': 'Srilankan Tamil',
        '3': 'Indian Tamil',
        '4': 'Srilankan Moor',
        '5': 'Burgher',
        '6': 'Malay',
        '7': 'Srilankan Chetty',
        '8': 'Bharatha',
        '9': 'Indian Moor',
        '10': 'Pakistan Moor',
        '11': 'Other Foreigners',
        '12': 'Other Srilankans'
    }
    
    # Mapping for text values
    text_map = {
        'Sinhalese': 'Sinhalese',
        'Srilankan Tamil': 'Srilankan Tamil',
        'Sri Lankan Tamil': 'Srilankan Tamil',
        'Indian Tamil': 'Indian Tamil',
        'Srilankan Moor': 'Srilankan Moor',
        'Sri Lankan Moor': 'Srilankan Moor',
        'Moor': 'Srilankan Moor',
        'Burgher': 'Burgher',
        'Malay': 'Malay',
        'Srilankan Chetty': 'Srilankan Chetty',
        'Bharatha': 'Bharatha',
        'Indian Moor': 'Indian Moor',
        'Pakistan Moor': 'Pakistan Moor',
        'Other Foreigners': 'Other Foreigners',
        'Other Srilankans': 'Other Srilankans'
    }
    
    # Check if it's a numeric code
    if race_str in code_map:
        return code_map[race_str]
    
    # Check if it's a text value
    return text_map.get(race_str, None)

# Create standardized ethnicity columns
df['Mother_Ethnicity_Std'] = df[MOTHER_RACE_COL].apply(standardize_ethnicity)
df['Father_Ethnicity_Std'] = df[FATHER_RACE_COL].apply(standardize_ethnicity)

# Show unique ethnicities found
unique_ethnicities = df['Mother_Ethnicity_Std'].dropna().unique()
print(f"\nUnique ethnicities found in dataset: {sorted(unique_ethnicities)}")

# ============================================
# STEP 3: DISTRICT-LEVEL MISSING BIRTH WEIGHT ANALYSIS
# ============================================

print("\n📊 STEP 3: District-Level Missing Birth Weight Analysis")
print("-" * 80)

# Get unique districts
districts = df[DISTRICT_COL].dropna().unique()
district_summary = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    
    total_births = len(district_data)
    # CHANGED: Now using Missing_Birth_Weight
    missing_weight = district_data['Missing_Birth_Weight'].sum()
    complete_weight = total_births - missing_weight
    
    district_summary.append({
        'District': district,
        'Total_Births': total_births,
        'Missing_Weight': missing_weight,
        'Complete_Weight': complete_weight,
        'Missing_Rate': (missing_weight / total_births * 100) if total_births > 0 else 0,
        # Percentage of total missing records contributed by this district
        'Pct_of_Total_Missing': (missing_weight / total_missing * 100) if total_missing > 0 else 0
    })

# Create district summary DataFrame
district_df = pd.DataFrame(district_summary)
district_df = district_df.sort_values('Missing_Rate', ascending=False)

# NEW: Get top 15 districts by missing rate for detailed analysis
TOP_N_DISTRICTS = 15
top_districts_list = district_df.head(TOP_N_DISTRICTS)['District'].tolist()
print(f"\nTop {TOP_N_DISTRICTS} districts with highest missing rates:")
for idx, (_, row) in enumerate(district_df.head(TOP_N_DISTRICTS).iterrows(), 1):
    print(f"   {idx}. {row['District']}: {row['Missing_Rate']:.2f}% ({row['Missing_Weight']:,} missing records)")

# ============================================
# STEP 4: ETHNICITY DISTRIBUTION BY DISTRICT (TOP 15 DISTRICTS ONLY)
# ============================================

print(f"\n📊 STEP 4: Ethnicity Distribution by District (Top {TOP_N_DISTRICTS} Districts by Missing Rate)")
print("-" * 80)

# Create detailed ethnicity-district analysis - ONLY FOR TOP 15 DISTRICTS
district_ethnicity_analysis = []

for district in top_districts_list:
    district_data = df[df[DISTRICT_COL] == district]
    district_total = len(district_data)
    district_missing_total = district_data['Missing_Birth_Weight'].sum()
    
    if district_total == 0:
        continue
    
    for ethnicity, config in ETHNICITIES.items():
        ethnic_mothers = district_data[district_data['Mother_Ethnicity_Std'] == ethnicity].shape[0]
        # CHANGED: Now using Missing_Birth_Weight
        ethnic_missing = district_data[
            (district_data['Mother_Ethnicity_Std'] == ethnicity) & 
            (district_data['Missing_Birth_Weight'] == True)
        ].shape[0]
        
        if ethnic_mothers > 0:
            pct_of_district = (ethnic_mothers / district_total * 100)
            pct_of_district_missing = (ethnic_missing / district_total * 100)
            missing_rate_in_ethnic = (ethnic_missing / ethnic_mothers * 100)
            
            # Percentage of this district's missing records contributed by this ethnicity
            pct_of_district_missing_records = (ethnic_missing / district_missing_total * 100) if district_missing_total > 0 else 0
            
            # Percentage of total national missing records contributed by this ethnicity in this district
            pct_of_national_missing = (ethnic_missing / total_missing * 100) if total_missing > 0 else 0
            
            district_ethnicity_analysis.append({
                'District': district,
                'Ethnicity': ethnicity,
                'IUPAC_Code': config['code'],
                'IUPAC_Name': config['iupac_name'],
                'Total_Mothers': ethnic_mothers,
                'Pct_of_District_Total': round(pct_of_district, 2),
                'Missing_Weight': ethnic_missing,  # CHANGED
                'Missing_Rate_in_Ethnic': round(missing_rate_in_ethnic, 2),
                'Pct_of_District_Missing': round(pct_of_district_missing, 2),
                # NEW METRICS:
                'Pct_of_District_Missing_Records': round(pct_of_district_missing_records, 2),  # % of this district's missing
                'Pct_of_National_Missing': round(pct_of_national_missing, 2)  # % of total national missing
            })

# Create DataFrame
if district_ethnicity_analysis:
    ethnicity_district_df = pd.DataFrame(district_ethnicity_analysis)
    
    # ============================================
    # CREATE WORD DOCUMENT WITH IUPAC STANDARDS
    # ============================================
    
    print("\n📄 Creating Word Document with IUPAC Standards...")
    
    # Create new document
    doc = Document()
    
    # Set document margins
    sections = doc.sections
    for section in sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)
    
    # Add title - CHANGED
    title = doc.add_heading('Missing Birth Weight Analysis Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add subtitle with IUPAC standards
    subtitle = doc.add_heading('Sri Lanka Vital Statistics - IUPAC Compliant Nomenclature', 2)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add date
    date_para = doc.add_paragraph(f'Report Generated: {datetime.now().strftime("%B %d, %Y")}')
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 1: EXECUTIVE SUMMARY
    # ============================================
    
    doc.add_heading('1. Executive Summary', level=1)
    
    # CHANGED: Updated summary text for birth weight
    summary_text = f"""
    This report presents a comprehensive analysis of missing birth weight data in Sri Lanka, 
    stratified by district and ethnicity. The analysis follows IUPAC (International Union of 
    Pure and Applied Chemistry) standards for ethnic nomenclature to ensure international 
    consistency and scientific rigor.
    
    Key Findings:
    • Total Records Analyzed: {total_records:,}
    • Missing Birth Weight Records: {total_missing:,} ({total_missing/total_records*100:.2f}%)
    • Complete Records: {total_complete:,} ({total_complete/total_records*100:.2f}%)
    • Number of Districts Analyzed: {len(districts)}
    • Number of Ethnic Groups Identified: {len(unique_ethnicities)}
    • Focus Districts: Top {TOP_N_DISTRICTS} districts with highest missing rates
    
    Note: Birth weight is a critical indicator for newborn health, neonatal mortality, and 
    long-term developmental outcomes. Complete birth weight data is essential for:
    • Low birth weight surveillance (<2500g)
    • Neonatal mortality risk assessment
    • Maternal nutrition program evaluation
    • Public health intervention planning
    """
    
    doc.add_paragraph(summary_text)
    
    # ============================================
    # SECTION 2: IUPAC ETHNICITY CLASSIFICATION
    # ============================================
    
    doc.add_heading('2. IUPAC Ethnicity Classification System', level=1)
    doc.add_paragraph('The following ethnicity codes follow IUPAC standards for population genetics and vital statistics reporting:')
    
    # Create ethnicity table
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Light Grid Accent 1'
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'IUPAC Standard Name'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
        row_cells[2].text = config['iupac_name']
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 3: DISTRICT-WISE ANALYSIS (TOP 15 DISTRICTS)
    # ============================================
    
    doc.add_heading(f'3. District-Wise Missing Birth Weight Analysis (Top {TOP_N_DISTRICTS} Districts by Missing Rate)', level=1)
    
    # Add district summary table - CHANGED with new metrics
    doc.add_heading('3.1 District Summary Statistics', level=2)
    
    district_table = doc.add_table(rows=1, cols=6)
    district_table.style = 'Light Grid Accent 1'
    hdr_cells = district_table.rows[0].cells
    hdr_cells[0].text = 'District'
    hdr_cells[1].text = 'Total Births'
    hdr_cells[2].text = 'Missing Weight Records'
    hdr_cells[3].text = 'Missing Rate (%)'
    hdr_cells[4].text = '% of Total Missing Records'
    hdr_cells[5].text = 'Rank by Missing Rate'
    
    # Show only top 15 districts
    district_by_rate = district_df.head(TOP_N_DISTRICTS)
    
    for idx, (_, row) in enumerate(district_by_rate.iterrows(), 1):
        row_cells = district_table.add_row().cells
        row_cells[0].text = row['District']
        row_cells[1].text = f"{row['Total_Births']:,}"
        row_cells[2].text = f"{row['Missing_Weight']:,}"
        row_cells[3].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[4].text = f"{row['Pct_of_Total_Missing']:.2f}%"
        row_cells[5].text = str(idx)
    
    # Add districts with highest missing counts among top 15
    doc.add_heading('3.2 Districts with Highest Missing Counts (Within Top 15 by Rate)', level=2)
    
    top_districts_by_count = district_by_rate.nlargest(10, 'Missing_Weight')
    count_table = doc.add_table(rows=1, cols=4)
    count_table.style = 'Light Grid Accent 1'
    hdr_cells = count_table.rows[0].cells
    hdr_cells[0].text = 'Rank by Count'
    hdr_cells[1].text = 'District'
    hdr_cells[2].text = 'Missing Weight Count'
    hdr_cells[3].text = 'Missing Rate (%)'
    
    for idx, (_, row) in enumerate(top_districts_by_count.iterrows(), 1):
        row_cells = count_table.add_row().cells
        row_cells[0].text = str(idx)
        row_cells[1].text = row['District']
        row_cells[2].text = f"{row['Missing_Weight']:,}"
        row_cells[3].text = f"{row['Missing_Rate']:.2f}%"
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 4: DETAILED DISTRICT-ETHNICITY ANALYSIS (TOP 15 DISTRICTS)
    # ============================================
    
    doc.add_heading(f'4. Detailed District-Ethnicity Analysis (Top {TOP_N_DISTRICTS} Districts by Missing Rate)', level=1)
    doc.add_paragraph(f'The following tables show the ethnicity distribution and missing birth weight patterns for the {TOP_N_DISTRICTS} districts with the highest missing rates, following IUPAC nomenclature.')
    
    for district in top_districts_list[:TOP_N_DISTRICTS]:
        district_ethnic = ethnicity_district_df[ethnicity_district_df['District'] == district]
        
        if len(district_ethnic) > 0:
            district_total = district_df[district_df['District'] == district]['Total_Births'].values[0]
            district_missing = district_df[district_df['District'] == district]['Missing_Weight'].values[0]
            district_pct_missing = district_df[district_df['District'] == district]['Pct_of_Total_Missing'].values[0]
            
            # Add district header
            doc.add_heading(f'{district}', level=2)
            doc.add_paragraph(f'Total Births: {district_total:,} | Missing Weight: {district_missing:,} ({district_missing/district_total*100:.2f}%) | {district_pct_missing:.1f}% of National Missing Records')
            
            # Create ethnicity table for district - CHANGED with new columns
            ethnic_table = doc.add_table(rows=1, cols=9)
            ethnic_table.style = 'Light Grid Accent 1'
            hdr_cells = ethnic_table.rows[0].cells
            hdr_cells[0].text = 'IUPAC Code'
            hdr_cells[1].text = 'Ethnicity'
            hdr_cells[2].text = 'Count'
            hdr_cells[3].text = '% of District'
            hdr_cells[4].text = 'Missing Weight'
            hdr_cells[5].text = 'Missing Rate (%)'
            hdr_cells[6].text = '% of District Missing Records'
            hdr_cells[7].text = '% of National Missing'
            hdr_cells[8].text = 'Rank in District'
            
            district_ethnic_sorted = district_ethnic.sort_values('Pct_of_District_Total', ascending=False)
            
            # Add rank by missing count within district
            district_ethnic_sorted = district_ethnic_sorted.copy()
            district_ethnic_sorted['Rank_In_District'] = district_ethnic_sorted['Missing_Weight'].rank(ascending=False, method='dense').astype(int)
            
            for _, row in district_ethnic_sorted.iterrows():
                row_cells = ethnic_table.add_row().cells
                row_cells[0].text = row['IUPAC_Code']
                row_cells[1].text = row['Ethnicity']
                row_cells[2].text = f"{row['Total_Mothers']:,}"
                row_cells[3].text = f"{row['Pct_of_District_Total']:.2f}%"
                row_cells[4].text = f"{row['Missing_Weight']:,}"
                row_cells[5].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[6].text = f"{row['Pct_of_District_Missing_Records']:.2f}%"
                row_cells[7].text = f"{row['Pct_of_National_Missing']:.2f}%"
                row_cells[8].text = str(row['Rank_In_District'])
            
            # Add district summary
            doc.add_paragraph()
            most_prevalent = district_ethnic_sorted.iloc[0]
            highest_missing = district_ethnic_sorted.loc[district_ethnic_sorted['Missing_Rate_in_Ethnic'].idxmax()]
            largest_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_District_Missing_Records'].idxmax()]
            largest_national_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_National_Missing'].idxmax()]
            
            summary_para = doc.add_paragraph()
            summary_para.add_run('District Summary:').bold = True
            doc.add_paragraph(f'• Most prevalent ethnicity: {most_prevalent["Ethnicity"]} ({most_prevalent["IUPAC_Code"]}) - {most_prevalent["Pct_of_District_Total"]:.1f}% of district')
            doc.add_paragraph(f'• Highest missing rate: {highest_missing["Ethnicity"]} ({highest_missing["IUPAC_Code"]}) - {highest_missing["Missing_Rate_in_Ethnic"]:.1f}% missing birth weight')
            doc.add_paragraph(f'• Largest contributor to district missing data: {largest_contributor["Ethnicity"]} ({largest_contributor["IUPAC_Code"]}) - {largest_contributor["Pct_of_District_Missing_Records"]:.1f}% of district missing records')
            doc.add_paragraph(f'• Largest contributor to national missing data: {largest_national_contributor["Ethnicity"]} ({largest_national_contributor["IUPAC_Code"]}) - {largest_national_contributor["Pct_of_National_Missing"]:.1f}% of national missing records')
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 5: ETHNICITY-SPECIFIC ANALYSIS
    # ============================================
    
    doc.add_heading('5. Ethnicity-Specific Analysis', level=1)
    
    # Calculate overall ethnicity statistics - CHANGED with new metrics
    ethnicity_overall = ethnicity_district_df.groupby(['Ethnicity', 'IUPAC_Code', 'IUPAC_Name']).agg({
        'Total_Mothers': 'sum',
        'Missing_Weight': 'sum',  # CHANGED
        'Pct_of_National_Missing': 'sum'  # NEW: Sum of national percentages
    }).reset_index()
    ethnicity_overall['Missing_Rate'] = (ethnicity_overall['Missing_Weight'] / ethnicity_overall['Total_Mothers'] * 100)
    ethnicity_overall = ethnicity_overall.sort_values('Missing_Rate', ascending=False)
    
    doc.add_heading('5.1 Overall Ethnicity Statistics', level=2)
    
    overall_table = doc.add_table(rows=1, cols=6)
    overall_table.style = 'Light Grid Accent 1'
    hdr_cells = overall_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'Total Births'
    hdr_cells[3].text = 'Missing Weight Records'
    hdr_cells[4].text = 'Missing Rate (%)'
    hdr_cells[5].text = '% of National Missing'
    
    for _, row in ethnicity_overall.iterrows():
        row_cells = overall_table.add_row().cells
        row_cells[0].text = row['IUPAC_Code']
        row_cells[1].text = row['Ethnicity']
        row_cells[2].text = f"{row['Total_Mothers']:,}"
        row_cells[3].text = f"{row['Missing_Weight']:,}"
        row_cells[4].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[5].text = f"{row['Pct_of_National_Missing']:.2f}%"
    
    # Add ethnicity-specific district analysis
    doc.add_heading('5.2 Ethnicity-Specific District Analysis (Top 15 Districts)', level=2)
    
    for ethnicity in list(ETHNICITIES.keys())[:6]:  # Show first 6 ethnicities
        ethnic_data = ethnicity_district_df[ethnicity_district_df['Ethnicity'] == ethnicity]
        
        if len(ethnic_data) > 0:
            ethnic_total = ethnic_data['Total_Mothers'].sum()
            ethnic_missing = ethnic_data['Missing_Weight'].sum()
            ethnic_rate = (ethnic_missing / ethnic_total * 100) if ethnic_total > 0 else 0
            ethnic_national_pct = ethnic_data['Pct_of_National_Missing'].sum()
            
            doc.add_heading(f'{ETHNICITIES[ethnicity]["full_name"]} ({ETHNICITIES[ethnicity]["code"]})', level=3)
            doc.add_paragraph(f'Overall Statistics: Total Births: {ethnic_total:,} | Missing Weight: {ethnic_missing:,} ({ethnic_rate:.2f}%) | {ethnic_national_pct:.1f}% of National Missing Records')
            
            # Create table for districts with highest missing rates AND highest missing counts for this ethnicity
            doc.add_paragraph('Districts with Highest Missing Rates for this Ethnicity:', style='List Bullet')
            rate_table = doc.add_table(rows=1, cols=5)
            rate_table.style = 'Light Grid Accent 1'
            hdr_cells = rate_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Weight'
            hdr_cells[3].text = 'Missing Rate (%)'
            hdr_cells[4].text = '% of Ethnic Missing'
            
            ethnic_sorted_by_rate = ethnic_data.sort_values('Missing_Rate_in_Ethnic', ascending=False).head(10)
            
            for _, row in ethnic_sorted_by_rate.iterrows():
                row_cells = rate_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Weight']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[4].text = f"{(row['Missing_Weight']/ethnic_missing*100):.2f}%"
            
            doc.add_paragraph()
            
            # NEW: Districts with highest missing counts for this ethnicity
            doc.add_paragraph('Districts with Highest Missing Counts for this Ethnicity:', style='List Bullet')
            count_table = doc.add_table(rows=1, cols=5)
            count_table.style = 'Light Grid Accent 1'
            hdr_cells = count_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Weight'
            hdr_cells[3].text = 'Missing Rate (%)'
            hdr_cells[4].text = '% of Ethnic Missing'
            
            ethnic_sorted_by_count = ethnic_data.sort_values('Missing_Weight', ascending=False).head(10)
            
            for _, row in ethnic_sorted_by_count.iterrows():
                row_cells = count_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Weight']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[4].text = f"{(row['Missing_Weight']/ethnic_missing*100):.2f}%"
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 6: CONCLUSIONS AND RECOMMENDATIONS
    # ============================================
    
    doc.add_heading('6. Conclusions and Recommendations', level=1)
    
    # Find key insights
    worst_district_by_rate = district_df.iloc[0]
    worst_district_by_count = district_df.nlargest(1, 'Missing_Weight').iloc[0]
    worst_ethnicity = ethnicity_overall.iloc[0] if len(ethnicity_overall) > 0 else None
    
    # Find which ethnicity contributes most to national missing
    top_national_contributor = ethnicity_overall.nlargest(1, 'Pct_of_National_Missing').iloc[0] if len(ethnicity_overall) > 0 else None
    
    # CHANGED: Updated conclusions for birth weight
    conclusions = f"""
    6.1 Key Findings
    
    • Data Completeness: {total_complete/total_records*100:.2f}% of birth records have complete birth weight data,
      indicating {total_missing/total_records*100:.2f}% of records require data quality improvement.
    
    • Geographic Disparities by Rate: {worst_district_by_rate['District']} shows the highest missing rate for birth weight at 
      {worst_district_by_rate['Missing_Rate']:.2f}%, suggesting potential data collection challenges in this region.
    
    • Geographic Disparities by Count: {worst_district_by_count['District']} has the highest absolute number of missing records
      ({worst_district_by_count['Missing_Weight']:,} records, representing {worst_district_by_count['Pct_of_Total_Missing']:.1f}% of all missing data).
    """
    
    if worst_ethnicity is not None:
        conclusions += f"""
    • Ethnic Disparities by Rate: {worst_ethnicity['Ethnicity']} ({worst_ethnicity['IUPAC_Code']}) has the highest 
      missing rate for birth weight at {worst_ethnicity['Missing_Rate']:.2f}%, indicating potential systematic bias 
      in data collection across ethnic groups.
        """
    
    if top_national_contributor is not None:
        conclusions += f"""
    • Ethnic Disparities by Contribution: {top_national_contributor['Ethnicity']} ({top_national_contributor['IUPAC_Code']}) 
      contributes the largest share ({top_national_contributor['Pct_of_National_Missing']:.1f}%) of all missing birth weight records nationally.
        """
    
    conclusions += """
    • Public Health Implications: Missing birth weight data affects the accuracy of low birth weight surveillance, 
      neonatal mortality assessments, and maternal and child health program evaluations. Birth weight is a critical 
      indicator for:
      - Low birth weight (<2500g) prevalence monitoring
      - Neonatal mortality risk assessment
      - Maternal nutrition intervention effectiveness
      - Newborn health outcomes tracking
    
    6.2 Recommendations
    
    1. Prioritize High-Volume Districts: Focus data quality improvement efforts first on districts with the highest
       absolute missing counts for birth weight.
    """
    
    conclusions += f"""
    2. Target High-Rate Districts: Implement specialized interventions in districts with the highest missing rates
       ({worst_district_by_rate['District']}: {worst_district_by_rate['Missing_Rate']:.1f}% missing).
    """
    
    if worst_ethnicity is not None:
        conclusions += f"""
    3. Ethnicity-Specific Interventions: Develop targeted data quality improvement programs for ethnic groups 
       with both high missing rates ({worst_ethnicity['Ethnicity']}: {worst_ethnicity['Missing_Rate']:.1f}%).
        """
    
    conclusions += """
    4. IUPAC Compliance: Maintain adherence to IUPAC ethnic nomenclature standards to ensure international 
       comparability and scientific rigor.
    
    5. Regular Monitoring: Establish quarterly data quality monitoring systems to track improvements in 
       missing birth weight rates by district and ethnicity, focusing on both rates and absolute counts.
    
    6. Capacity Building: Provide training for healthcare workers on the importance of accurate birth weight 
       measurement and documentation, especially in high-volume and high-rate districts.
    
    7. Equipment Maintenance: Ensure regular calibration and maintenance of weighing scales in healthcare 
       facilities to prevent equipment-related missing data.
    
    8. Electronic Health Records: Implement or strengthen electronic health record systems with mandatory 
       birth weight fields to reduce missing data.
    """
    
    doc.add_paragraph(conclusions)
    
    # Add methodology section
    doc.add_heading('7. Methodology', level=1)
    
    methodology = f"""
    This analysis was conducted using vital statistics data from Sri Lanka. The methodology follows 
    IUPAC standards for ethnic classification and WHO guidelines for demographic data collection.
    
    Data Sources:
    • Birth Registration Records: {total_records:,} records
    • Time Period: Full dataset analysis
    • Geographic Coverage: {len(districts)} districts
    • Focus Districts: Top {TOP_N_DISTRICTS} districts with highest missing rates
    
    Key Metrics Defined:
    
    1. Missing Rate: Percentage of records missing birth weight within a specific group
       Formula: (Missing Weight / Total Births) × 100
    
    2. Percentage of Total Missing Records: What proportion of ALL national missing records 
       comes from a specific district or ethnicity
       Formula: (Missing Weight in Group / Total National Missing) × 100
    
    3. Percentage of District Missing Records: What proportion of a district's missing records 
       comes from a specific ethnicity
       Formula: (Missing Weight for Ethnicity in District / Total District Missing) × 100
    
    4. Percentage of National Missing: What proportion of ALL national missing records comes 
       from a specific ethnicity-district combination
    
    Birth Weight Definition:
    Birth weight is the first weight of the newborn taken within the first hours of life, 
    measured in grams. Complete birth weight data is essential for:
    • Low birth weight surveillance (<2500g)
    • Very low birth weight tracking (<1500g)
    • Extremely low birth weight monitoring (<1000g)
    • Neonatal mortality risk assessment
    • Maternal nutrition program evaluation
    • Demographic transition analysis
    • Newborn health outcomes research
    
    Birth Weight Categories (WHO):
    • Low Birth Weight: <2500g
    • Normal Birth Weight: 2500g - 3999g
    • High Birth Weight: ≥4000g
    
    Ethnic Classification:
    Ethnicities were classified according to IUPAC standards using the following codes:
    """
    
    doc.add_paragraph(methodology)
    
    # Add IUPAC code reference
    ref_table = doc.add_table(rows=1, cols=2)
    ref_table.style = 'Light Grid Accent 1'
    hdr_cells = ref_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnic Group'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = ref_table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
    
    # Save the document - CHANGED filename
    filename = f'Missing_Birth_Weight_Analysis_IUPAC_{datetime.now().strftime("%Y%m%d_%H%M%S")}.docx'
    doc.save(filename)
    print(f"\n✅ Word document saved as: {filename}")
    
    # ============================================
    # STEP 5: DISPLAY SUMMARY IN CONSOLE
    # ============================================
    
    print("\n" + "=" * 100)
    print(f"📊 FINAL SUMMARY: Missing Birth Weight Analysis by District and Ethnicity (Top {TOP_N_DISTRICTS} Districts)")
    print("=" * 100)
    
    print(f"\n✅ Analysis Complete!")
    print(f"✅ Word Document Generated: {filename}")
    print(f"\nOverall Statistics:")
    print(f"   • Total Districts: {len(districts)}")
    print(f"   • Focus Districts: {TOP_N_DISTRICTS} (highest missing rates)")
    print(f"   • Total Births: {total_records:,}")
    print(f"   • Total Missing Birth Weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    
    print(f"\n🏆 Top {TOP_N_DISTRICTS} Districts with Highest Missing RATE:")
    for idx, (_, row) in enumerate(district_df.head(TOP_N_DISTRICTS).iterrows(), 1):
        print(f"   {idx}. {row['District']}: {row['Missing_Rate']:.2f}% ({row['Missing_Weight']:,}/{row['Total_Births']:,})")
    
    print(f"\n🏆 Top 5 Districts with Highest Missing COUNT (absolute) among top {TOP_N_DISTRICTS}:")
    district_by_count = district_by_rate.nlargest(5, 'Missing_Weight')
    for _, row in district_by_count.iterrows():
        print(f"   • {row['District']}: {row['Missing_Weight']:,} missing records ({row['Missing_Rate']:.2f}% of district)")
    
    print(f"\n🏆 Top 5 Districts by % of Total National Missing among top {TOP_N_DISTRICTS}:")
    district_by_pct = district_by_rate.nlargest(5, 'Pct_of_Total_Missing')
    for _, row in district_by_pct.iterrows():
        print(f"   • {row['District']}: {row['Pct_of_Total_Missing']:.2f}% of national missing ({row['Missing_Weight']:,} records)")
    
    if len(ethnicity_overall) > 0:
        print(f"\n🏆 Top 5 Ethnicities with Highest Missing RATE (Overall):")
        for _, row in ethnicity_overall.head(5).iterrows():
            print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Missing_Rate']:.2f}% ({row['Missing_Weight']:,}/{row['Total_Mothers']:,})")
        
        print(f"\n🏆 Top 5 Ethnicities by % of Total National Missing:")
        ethnicity_by_pct = ethnicity_overall.nlargest(5, 'Pct_of_National_Missing')
        for _, row in ethnicity_by_pct.iterrows():
            print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Pct_of_National_Missing']:.2f}% of national missing ({row['Missing_Weight']:,} records)")

else:
    print("\n⚠️ No ethnicity data found. Please check the ethnicity column values.")
    print(f"Sample values from {MOTHER_RACE_COL}:")
    print(df[MOTHER_RACE_COL].value_counts().head(10))

print("\n✅ Analysis Complete!")

🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY

📊 STEP 1: Missing Birth Weight Analysis
--------------------------------------------------------------------------------

Overall Statistics:
   • Total records: 301,712
   • Missing birth weight: 33,542 (11.12%)
   • Complete birth weight: 268,170 (88.88%)

📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)
--------------------------------------------------------------------------------

Unique ethnicities found in dataset: ['Burgher', 'Indian Tamil', 'Malay', 'Sinhalese', 'Srilankan Moor', 'Srilankan Tamil']

📊 STEP 3: District-Level Missing Birth Weight Analysis
--------------------------------------------------------------------------------

Top 15 districts with highest missing rates:
   1. Batticaloa: 79.98% (7,651 missing records)
   2. Ampara: 41.57% (6,039 missing records)
   3. Vavuniya: 13.60% (509 missing records)
   4. Gampaha: 12.35% (2,378 missing records)
   5. Colombo: 12.27% (5,243 missing records)
   6. Mu